In [138]:
# --- CÉLULA DE CONFIGURAÇÃO PRINCIPAL ---
import os # Garante que 'os' está importado

# --- PARÂMETROS DE EXECUÇÃO ---
# Altere esta variável para 'UFPB' ou 'UFCG' para rodar a análise
ORGAO_NOME_CURTO = 'UFCG' 
ano = 2025 # Ano a ser processado
# -----------------------------------

# Define o filtro e nomes de arquivo com base na instituição
if ORGAO_NOME_CURTO == 'UFPB':
    ORGAO_FILTRO_STR = 'UFPB|UNIVERSIDADE FEDERAL DA PARAIBA|UNIVERSIDADE FEDERAL DA PARAÍBA'
elif ORGAO_NOME_CURTO == 'UFCG':
    ORGAO_FILTRO_STR = 'UFCG|UNIVERSIDADE FEDERAL DE CAMPINA GRANDE'
else:
    ORGAO_FILTRO_STR = ORGAO_NOME_CURTO # Fallback para outros nomes

print(f"--- Iniciando processamento para: {ORGAO_NOME_CURTO} (Ano: {ano}) ---")

# Define nomes de pastas e arquivos de SAÍDA
PASTA_SAIDA_ANO = f'dadosViagens/dados_viagens{ano}'
if not os.path.exists(PASTA_SAIDA_ANO):
    os.makedirs(PASTA_SAIDA_ANO)
    
# Arquivos de Saída Opcionais (gerados pelo notebook)
ARQUIVO_LOOKUP_OUT = os.path.join(PASTA_SAIDA_ANO, f'lookup_distancias_unicas_{ORGAO_NOME_CURTO}_aereo_{ano}.csv')
ARQUIVO_ITINERARIOS_OUT_CSV = os.path.join(PASTA_SAIDA_ANO, f'itinerarios_detalhados_{ORGAO_NOME_CURTO}_aereo_{ano}.csv')
ARQUIVO_ITINERARIOS_OUT_JSON = os.path.join(PASTA_SAIDA_ANO, f'itinerarios_detalhados_{ORGAO_NOME_CURTO}_aereo_{ano}.json')

# Arquivo Mestre (Entrada para os notebooks de métricas)
ARQUIVO_MASTER_OUT = os.path.join(PASTA_SAIDA_ANO, f'df_master_{ORGAO_NOME_CURTO}_aereo_{ano}.csv')

# Arquivos de Comparação (da Etapa 6 / Célula 14)
ARQUIVO_COMPLETO_OUT = f'diferenca_FINAL_completa_{ORGAO_NOME_CURTO}.csv'
ARQUIVO_DISCREPANCIAS_OUT = f'diferenca_SIGNIFICATIVAS_{ORGAO_NOME_CURTO}.csv'
ARQUIVO_NAO_ENCONTRADOS_OUT = f'viagens_nao_encontradas_no_baseline_{ORGAO_NOME_CURTO}.csv'

--- Iniciando processamento para: UFCG (Ano: 2025) ---


In [139]:
# Nomes das colunas de interesse nos dataframes
COL_ID_TRECHO = 'Identificador do processo de viagem'
COL_ID_TRECHO_SPACED = 'Identificador do processo de viagem '
COL_ORIGEM = 'Origem - Cidade'
COL_DESTINO = 'Destino - Cidade'

# Importa a classe Viagens do módulo viagens_dowload (verifique se o arquivo .py está na mesma pasta)
try:
    from viagens_dowload import Viagens
except ModuleNotFoundError:
    print("❌ ERRO: Arquivo 'viagens_dowload.py' não encontrado. Certifique-se de que ele está na mesma pasta do notebook.")
    # Adicione código para parar a execução se necessário, ou lide com o erro.
    # exit() # Exemplo para parar

ano = 2023
vi = Viagens(ano)

import pandas as pd
import numpy as np
import os # Importa os aqui para uso posterior

# Carregar Arquivos CSV

In [140]:
# para fazer o download
# viagem_df, trecho_df, passagem_df = vi.pegarViagens()

# Se já tiver os csvs baixados, só carrega os dataframes
viagem_df, trecho_df, passagem_df = vi.carregar_csvs()

✅ 2023_Passagem.csv carregado com 390728 linhas (sem cabeçalho).
✅ 2023_Trecho.csv carregado com 1817649 linhas (sem cabeçalho).
✅ 2023_Viagem.csv carregado com 832499 linhas (sem cabeçalho).


In [141]:
# corrigir a coluna indentificador do processo de viagem que tem um espaço a mais no nome
trecho_df.rename(columns={COL_ID_TRECHO_SPACED: COL_ID_TRECHO}, inplace=True)   


# Limpeza e Preparação dos Dados

In [142]:
# --- CÉLULA 3: FILTRAGEM INICIAL (COM VALIDAÇÃO DE INTEGRIDADE) ---
import pandas as pd
import numpy as np

print("\n--- Iniciando Filtragem Inicial ---")

# --- 1. Verificação de Integridade (Remover Órfãos) ---
print("🔄 Verificando integridade dos dados (removendo trechos/viagens órfãos)...")

try:
    # Garante que os DataFrames existem
    if 'viagem_df' not in locals() or 'passagem_df' not in locals() or 'trecho_df' not in locals():
        raise NameError("DataFrames originais (viagem_df, passagem_df, trecho_df) não foram carregados.")

    # Renomeia a coluna com espaço em trecho_df (necessário ANTES de obter IDs)
    if COL_ID_TRECHO_SPACED in trecho_df.columns:
        trecho_df = trecho_df.rename(columns={COL_ID_TRECHO_SPACED: COL_ID_TRECHO})
        print("   - Coluna de ID com espaço em trecho_df renomeada.")

    # Limpa os IDs de todos os DataFrames
    viagem_df[COL_ID_TRECHO] = viagem_df[COL_ID_TRECHO].astype(str).str.strip().str.zfill(21)
    passagem_df[COL_ID_TRECHO] = passagem_df[COL_ID_TRECHO].astype(str).str.strip().str.zfill(21)
    trecho_df[COL_ID_TRECHO] = trecho_df[COL_ID_TRECHO].astype(str).str.strip().str.zfill(21)

    # Obtém conjuntos (sets) de IDs únicos de cada arquivo
    ids_viagem = set(viagem_df[COL_ID_TRECHO].unique())
    ids_passagem = set(passagem_df[COL_ID_TRECHO].unique())
    ids_trecho = set(trecho_df[COL_ID_TRECHO].unique())

    print(f"   - IDs únicos encontrados em viagem_df: {len(ids_viagem)}")
    print(f"   - IDs únicos encontrados em passagem_df: {len(ids_passagem)}")
    print(f"   - IDs únicos encontrados em trecho_df: {len(ids_trecho)}")

    # Encontra a interseção: IDs que existem em TODOS OS TRÊS arquivos
    ids_validos = ids_viagem.intersection(ids_passagem).intersection(ids_trecho)
    
    total_validos = len(ids_validos)
    print(f"✅ Encontrados {total_validos} IDs de viagem válidos (presentes em viagem, passagem e trecho).")

    if total_validos == 0:
        print("❌ ERRO CRÍTICO: Nenhum ID de viagem em comum encontrado nos três arquivos. Verifique os dados de entrada.")
        # Decide se para a execução
        # exit() 
    else:
        # Filtra os DataFrames originais para conter APENAS IDs válidos
        viagem_df = viagem_df[viagem_df[COL_ID_TRECHO].isin(ids_validos)].copy()
        passagem_df = passagem_df[passagem_df[COL_ID_TRECHO].isin(ids_validos)].copy()
        trecho_df = trecho_df[trecho_df[COL_ID_TRECHO].isin(ids_validos)].copy()
        print("✅ DataFrames principais filtrados para conter apenas IDs válidos (órfãos removidos).")

except NameError as e:
    print(f"❌ ERRO: {e}. Certifique-se de que a Célula 2 (Carregar Dados) foi executada.")
except Exception as e:
    print(f"❌ Ocorreu um erro inesperado na verificação de integridade: {e}")


# --- 2. Filtros Específicos (UFPB e Aéreo) ---
# (O código a partir daqui é o mesmo de antes, mas agora opera nos DataFrames já limpos)

if 'passagem_df' in locals() and 'trecho_df' in locals() and not passagem_df.empty and not trecho_df.empty:
    print("\n🔄 Aplicando filtros específicos (UFPB e Aéreo)...")
    
    # 2a. Filtro por Instituição (UFPB) - Identifica os IDs relevantes
    FILTRO_UFPB = 'UFPB|UNIVERSIDADE FEDERAL DA PARAIBA|UNIVERSIDADE FEDERAL DA PARAÍBA'
    NOME_COLUNA_ORGAO = 'Nome órgão solicitante' 

    NOME_COLUNA_ORGAO = 'Nome órgão solicitante' 
    if NOME_COLUNA_ORGAO in passagem_df.columns:
        passagem_df[NOME_COLUNA_ORGAO] = passagem_df[NOME_COLUNA_ORGAO].astype(str).str.strip()
        # <<< MUDANÇA AQUI >>>
        ufpb_ids_df = passagem_df[
            passagem_df[NOME_COLUNA_ORGAO].str.upper().str.contains(ORGAO_FILTRO_STR, na=False)
        ][[COL_ID_TRECHO]].drop_duplicates()
        
        print(
            f"- Encontrados {len(ufpb_ids_df)} IDs únicos de {ORGAO_NOME_CURTO} (dentro dos dados válidos).")
    else:
        print(f"❌ ERRO CRÍTICO: Coluna '{NOME_COLUNA_ORGAO}' não encontrada em 'passagem_df'.")
        ufpb_ids_df = pd.DataFrame(columns=[COL_ID_TRECHO]) # Cria DF vazio para evitar erro

    if ufpb_ids_df.empty:
        print("   ⚠️ Aviso: Nenhum ID de viagem da UFPB encontrado após filtros. O processamento continuará com dados vazios.")

    # 2b. Filtro por Meio de Transporte (Aéreo) no trecho_df
    trecho_df['Meio de transporte'] = trecho_df['Meio de transporte'].astype(str).str.strip()
    trecho_df_aereo = trecho_df[
        trecho_df['Meio de transporte'].str.upper() == 'AÉREO'
    ].copy()
    print(f"   - Encontrados {len(trecho_df_aereo)} trechos aéreos (dentro dos dados válidos).")

    # 2c. Combina os filtros: Mantém apenas trechos aéreos que pertencem aos IDs da UFPB
    if not ufpb_ids_df.empty and not trecho_df_aereo.empty:
        # IDs em trecho_df_aereo e ufpb_ids_df já estão limpos da Etapa 1
        trecho_df_filtrado = pd.merge(
            trecho_df_aereo,
            ufpb_ids_df,
            on=COL_ID_TRECHO,
            how='inner' 
        )
        # Substitui o trecho_df original pelo filtrado
        trecho_df = trecho_df_filtrado.copy()
        print(f"✅ Filtragem específica concluída. 'trecho_df' agora contém {len(trecho_df)} trechos (apenas aéreos da UFPB válidos).")
    else:
        # Se UFPB ou Aéreo estiverem vazios, o resultado final é vazio
        trecho_df = pd.DataFrame(columns=trecho_df.columns) # Cria um DF vazio com as mesmas colunas
        print("   ⚠️ 'trecho_df' final está vazio (nenhuma viagem aérea da UFPB encontrada nos dados válidos).")

else:
    print("⚠️ DataFrames não estão prontos, filtros específicos (UFPB, Aéreo) pulados.")

print("--- Fim da Filtragem Inicial ---")


--- Iniciando Filtragem Inicial ---
🔄 Verificando integridade dos dados (removendo trechos/viagens órfãos)...
   - IDs únicos encontrados em viagem_df: 218180
   - IDs únicos encontrados em passagem_df: 832497
   - IDs únicos encontrados em trecho_df: 714014
✅ Encontrados 218180 IDs de viagem válidos (presentes em viagem, passagem e trecho).
✅ DataFrames principais filtrados para conter apenas IDs válidos (órfãos removidos).

🔄 Aplicando filtros específicos (UFPB e Aéreo)...
- Encontrados 320 IDs únicos de UFCG (dentro dos dados válidos).
   - Encontrados 415222 trechos aéreos (dentro dos dados válidos).
✅ Filtragem específica concluída. 'trecho_df' agora contém 653 trechos (apenas aéreos da UFPB válidos).
--- Fim da Filtragem Inicial ---


# Filtros 

In [143]:
# --- CÉLULA 4: LIMPEZA BÁSICA DE CIDADES (STRIP) ---


print("\n--- Iniciando Limpeza Básica de Cidades ---")

try:
    # Verifica se o trecho_df existe da célula anterior
    if 'trecho_df' not in locals() or trecho_df.empty:
        # Se estiver vazio (nenhuma viagem UFPB Aérea Válida), apenas informa
        if 'trecho_df' in locals() and trecho_df.empty:
            print("✅ 'trecho_df' está vazio (nenhuma viagem UFPB Aérea Válida). Limpeza pulada.")
        else:
            raise NameError("'trecho_df' não encontrado. Execute a Célula 3 primeiro.")
    
    # Prossegue apenas se o DataFrame não estiver vazio
    if not trecho_df.empty:
        if COL_ORIGEM in trecho_df.columns and COL_DESTINO in trecho_df.columns:
            trecho_df[COL_ORIGEM] = trecho_df[COL_ORIGEM].astype(str).str.strip()
            trecho_df[COL_DESTINO] = trecho_df[COL_DESTINO].astype(str).str.strip()
            print("✅ Colunas de Origem/Destino limpas (str.strip()).")
        else:
            print(f"❌ Erro: Colunas {COL_ORIGEM} ou {COL_DESTINO} não encontradas.")
            
except Exception as e:
    print(f"❌ Erro na Célula 4 (Limpeza Básica): {e}")


--- Iniciando Limpeza Básica de Cidades ---
✅ Colunas de Origem/Destino limpas (str.strip()).


# Correções Manuais de Grafia de Cidades

In [144]:
# --- Limpeza Pós-Filtro ---
print("\n--- Iniciando Limpeza Pós-Filtro ---")

# 1. Garante limpeza das colunas de cidade no trecho_df filtrado
if COL_ORIGEM in trecho_df.columns and COL_DESTINO in trecho_df.columns:
    trecho_df[COL_ORIGEM] = trecho_df[COL_ORIGEM].astype(str).str.strip()
    trecho_df[COL_DESTINO] = trecho_df[COL_DESTINO].astype(str).str.strip()
    print("✅ Colunas de Origem/Destino limpas.")
else:
    print(f"❌ Erro: Colunas {COL_ORIGEM} ou {COL_DESTINO} não encontradas no trecho_df filtrado.")

# 2. Aplica Correções Manuais de Grafia (o dicionário 'correcoes' deve ter sido definido antes)
print("🔄 Aplicando correções manuais de grafia...")
try:
    correcoes = {
        # Cole aqui o dicionário 'correcoes' completo da sua versão anterior
        # Exemplo:
        'Palma de Malorca': 'Palma de Mallorca',
        'Wroc?aw': 'Wroclaw',
        # ... (todas as outras correções) ...
        'AssunÃ§Ã£o': 'Assunção',
        'NiterÃ³i': 'Niterói',
         'MacapÃ¡ - ArquipÃ©lago do Bailique': 'Macapá', # Simplificando
         'MacapÃ¡': 'Macapá',
         'VitÃ³ria': 'Vitória',
         'UÃ¡di Natrum': 'Wadi El Natrun, Egito', # Nome fonético correto
         'UberlÃ¢ndia': 'Uberlândia',
         'TÃ³quio': 'Tóquio',
         'TetuÃ£o': 'Tétouan, Marrocos', # Nome internacional + país
         'SÃ£o Paulo': 'São Paulo',
         'SÃ£o LuÃ\\xads': 'São Luís',
         'Santa Helena do UairÃ©n': 'Santa Elena de Uairén', # Correção geográfica
         'Cartagena das Ã\\x8dndias': 'Cartagena, Colômbia', # Adicionando país
         'CantÃ£o de Soleura': 'Solothurn, Suíça', # Nome internacional
         'BrasÃ\\xadlia': 'Brasília',
         'BogotÃ¡': 'Bogotá',
         'BelÃ©m': 'Belém',
         'GoiÃ¢nia': 'Goiânia',
         'Foz do IguaÃ§u': 'Foz do Iguaçu',
         'Fort Benning South - GeÃ³rgia': 'Fort Moore, GA, EUA', # Nome atual
         'FlorianÃ³polis': 'Florianópolis',
         'CosenÃ§a': 'Cosenza', # Correção para Italiano
         'Cidade do PanamÃ¡': 'Cidade do Panamá',
         'B?dlewo': 'Bydlewo, Polônia', # Adicionando país
         'Honj?': 'Honjo',
         'AnaheimCA - CalifÃ³rnia': 'Anaheim, Califórnia ', # Corrigido e simplificado
         'Serinagar': 'Srinagar',
         'Islascanarias': 'Ilhas Canárias',
         'Bangcoc': 'Bangkok',
         'Marraqueche': 'Marrakech',
         'Ponta Arenas': 'Punta Arenas',
         'Distrito de Yguazu': 'Minga Guazú, Paraguai', # Nome de localidade
         'Ludlow - Massachussets': 'Ludlow, Massachusetts', # Removendo hífen
         'Vilamontes': 'Villamontes', # Correção de digitação
         'Sophia-Atipolis': 'Sophia Antipolis', # Removendo hífen
         'Itajuipi': 'Itajuípe, Bahia', # Corrigindo e adicionando estado
         'Barcelona': 'Barcelona, Espanha', # Adicionando país
         'AnaheimCA - Califórnia': 'Anaheim, Califórnia', # Corrigido e simplificado
         'Pirenópolis': 'Pirenópolis, Goiás', # Adicionando estado
         'Piren\u00f3polis': 'Pirenópolis, Goiás', # Adicionando estado
         'pire': 'Pirenópolis, Goiás', # Corrigindo minúscula
         'Boa Vista': 'Boa Vista, Roraima',
         
         # --- ESPECIFICAÇÃO DAS CAPITAIS BRASILEIRAS ---
        'Aracaju': 'Aracaju, Sergipe',
        'Belém': 'Belém, Pará',
        'Belo Horizonte': 'Belo Horizonte, Minas Gerais',
        'Boa Vista': 'Boa Vista, Roraima', # Garante que prevaleça sobre a correção anterior, se necessário
        'Brasília': 'Brasília, Distrito Federal',
        'Campo Grande': 'Campo Grande, Mato Grosso do Sul',
        'Cuiabá': 'Cuiabá, Mato Grosso',
        'Curitiba': 'Curitiba, Paraná',
        'Florianópolis': 'Florianópolis, Santa Catarina',
        'Fortaleza': 'Fortaleza, Ceará', # Garante que prevaleça
        'Goiânia': 'Goiânia, Goiás',
        'João Pessoa': 'João Pessoa, Paraíba',
        'Macapá': 'Macapá, Amapá',
        'Maceió': 'Maceió, Alagoas',
        'Manaus': 'Manaus, Amazonas',
        'Natal': 'Natal, Rio Grande do Norte',
        'Palmas': 'Palmas, Tocantins',
        'Porto Alegre': 'Porto Alegre, Rio Grande do Sul',
        'Porto Velho': 'Porto Velho, Rondônia',
        'Recife': 'Recife, Pernambuco',
        'Rio Branco': 'Rio Branco, Acre',
        'Rio de Janeiro': 'Rio de Janeiro, Rio de Janeiro',
        'Salvador': 'Salvador, Bahia',
        'São Luís': 'São Luís, Maranhão',
        'São Paulo': 'São Paulo, São Paulo',
        'Teresina': 'Teresina, Piauí',
        'Vitória': 'Vitória, Espírito Santo',
        
        'Santiago': 'Santiago, Chile',
        'Cascavel': 'Cascavel, Paraná',
    }
    COLUNAS_CIDADE = [COL_ORIGEM, COL_DESTINO]
    for col in COLUNAS_CIDADE:
        if col in trecho_df.columns:
            trecho_df[col] = trecho_df[col].replace(correcoes)
            trecho_df[col] = trecho_df[col].astype(str).str.strip() # Strip extra pós-replace
    print("✅ Correções manuais aplicadas.")
except NameError:
    print("❌ ERRO: O dicionário 'correcoes' não foi definido. Cole-o na seção apropriada.")
except Exception as e:
    print(f"❌ Erro ao aplicar correções manuais: {e}")

print("--- Fim da Limpeza Pós-Filtro ---")


--- Iniciando Limpeza Pós-Filtro ---
✅ Colunas de Origem/Destino limpas.
🔄 Aplicando correções manuais de grafia...
✅ Correções manuais aplicadas.
--- Fim da Limpeza Pós-Filtro ---


In [145]:
# # --- CÉLULA 6 (REVISADA): CORREÇÕES MANUAIS E FILTRO O!=D ---
# import pandas as pd

# print("\n--- Aplicando correções manuais (dicionário) ---")

# try:
#     if 'trecho_df' not in locals():
#          raise NameError("'trecho_df' não encontrado. Execute a Célula 4 primeiro.")

#     if not trecho_df.empty:
#         # 1. Dicionário de Correções
#         correcoes = {
#             # --- CORREÇÕES DE AMBIGUIDADE (Importante) ---
#             'Boa Vista': 'Boa Vista, Roraima',
#             'Fortaleza': 'Fortaleza, Ceará',
#             'Santiago': 'Santiago de Compostela, Espanha',
#             'Cascavel': 'Cascavel, Paraná',
            
#             # --- ESPECIFICAÇÃO DAS CAPITAIS BRASILEIRAS ---
#             'Aracaju': 'Aracaju, Sergipe', 'Belém': 'Belém, Pará',
#             'Belo Horizonte': 'Belo Horizonte, Minas Gerais',
#             'Brasília': 'Brasília, Distrito Federal',
#             'Campo Grande': 'Campo Grande, Mato Grosso do Sul',
#             'Cuiabá': 'Cuiabá, Mato Grosso', 'Curitiba': 'Curitiba, Paraná',
#             'Florianópolis': 'Florianópolis, Santa Catarina',
#             'Goiânia': 'Goiânia, Goiás', 'João Pessoa': 'João Pessoa, Paraíba',
#             'Macapá': 'Macapá, Amapá', 'Maceió': 'Maceió, Alagoas',
#             'Manaus': 'Manaus, Amazonas', 'Natal': 'Natal, Rio Grande do Norte',
#             'Palmas': 'Palmas, Tocantins', 'Porto Alegre': 'Porto Alegre, Rio Grande do Sul',
#             'Porto Velho': 'Porto Velho, Rondônia', 'Recife': 'Recife, Pernambuco',
#             'Rio Branco': 'Rio Branco, Acre', 'Rio de Janeiro': 'Rio de Janeiro, Rio de Janeiro',
#             'Salvador': 'Salvador, Bahia', 'São Luís': 'São Luís, Maranhão',
#             'São Paulo': 'São Paulo, São Paulo', 'Teresina': 'Teresina, Piauí',
#             'Vitória': 'Vitória, Espírito Santo',

#             # --- CORREÇÕES ANTERIORES (Encoding, etc.) ---
#             # (Cole aqui o restante das suas correções de encoding, ex: 'NiterÃ³i': 'Niterói', etc.)
#             'Palma de Malorca': 'Palma de Mallorca', 'Wroc?aw': 'Wroclaw',
#             'Serinagar': 'Srinagar', 'Islascanarias': 'Ilhas Canárias',
#             'Bangcoc': 'Bangkok', 'Marraqueche': 'Marrakech',
#             # ... (Resto do dicionário) ...
#             'B?dlewo': 'Bydlewo, Polônia',
#             'Honj?': 'Honjo'
#         }

#         # 2. Aplica as Correções
#         COLUNAS_CIDADE = [COL_ORIGEM, COL_DESTINO]
#         for col in COLUNAS_CIDADE:
#             if col in trecho_df.columns:
#                 trecho_df[col] = trecho_df[col].replace(correcoes)
#                 trecho_df[col] = trecho_df[col].astype(str).str.strip() # Strip final
#         print("✅ Correções manuais (dicionário) aplicadas.")
        
#         # 3. FILTRO O!=D (APLICADO AGORA, DEPOIS DAS CORREÇÕES)
#         linhas_antes_filtro_od = len(trecho_df)
#         trecho_df = trecho_df[trecho_df[COL_ORIGEM] != trecho_df[COL_DESTINO]].copy()
#         linhas_removidas = linhas_antes_filtro_od - len(trecho_df)
        
#         if linhas_removidas > 0:
#             print(f"✅ Filtro O=D: Removidos {linhas_removidas} trechos onde Origem (corrigida) é igual a Destino (corrigido).")
#         else:
#             print("✅ Filtro O=D: Nenhum trecho com Origem=Destino encontrado (após correções).")
#         print(f"   - Trechos restantes para processar: {len(trecho_df)}")
        
#     else:
#         print("✅ 'trecho_df' está vazio. Correções manuais e filtro O=D pulados.")

# except Exception as e:
#     print(f"❌ Erro na Célula 6 (Correções Manuais / Filtro O=D): {e}")

# print("--- Fim das Correções e Filtro O=D ---")

In [146]:
# --- CÉLULA 5: FILTRO DE TRECHOS INVÁLIDOS (ORIGEM = DESTINO) ---
print("\n--- Aplicando Filtro (Origem != Destino) ---")

try:
    if 'trecho_df' not in locals():
         raise NameError("'trecho_df' não encontrado. Execute a Célula 4 primeiro.")

    if not trecho_df.empty:
        linhas_antes_filtro_od = len(trecho_df)
        # Remove trechos onde a origem e o destino são idênticos (após limpeza)
        trecho_df = trecho_df[trecho_df[COL_ORIGEM] != trecho_df[COL_DESTINO]].copy()
        linhas_removidas = linhas_antes_filtro_od - len(trecho_df)
        
        if linhas_removidas > 0:
            print(f"✅ Filtro O=D: Removidos {linhas_removidas} trechos onde Origem é igual a Destino.")
        else:
            print("✅ Filtro O=D: Nenhum trecho com Origem=Destino encontrado.")
        print(f"   - Trechos restantes para processar: {len(trecho_df)}")
    else:
        print("✅ 'trecho_df' já está vazio. Filtro O=D pulado.")
            
except Exception as e:
    print(f"❌ Erro na Célula 5 (Filtro O=D): {e}")


--- Aplicando Filtro (Origem != Destino) ---
✅ Filtro O=D: Nenhum trecho com Origem=Destino encontrado.
   - Trechos restantes para processar: 653


# Etapa 1: Cálculo de Distância por Trecho

In [147]:
# --- ETAPA 5 (REVISADA): CRIAR/ATUALIZAR LOOKUP MESTRE PERSISTENTE ---
import pandas as pd
import numpy as np
import os
# Certifique-se de que a classe Distancia está disponível
try:
    # Use o nome correto do seu arquivo .py
    from distanciaCidades2 import Distancia 
except ModuleNotFoundError:
     print("❌ ERRO: Arquivo 'distanciaCidades2_corrigido.py' não encontrado.")
     # Forçar parada ou tratar o erro, pois a célula não pode continuar
     # exit()
     raise # Re-levanta o erro para parar a execução da célula

print("\n--- Iniciando Criação/Atualização da Tabela Lookup Mestre Persistente ---")

# --- Constantes ---
# Nome do arquivo mestre (será salvo na mesma pasta do notebook)
MASTER_LOOKUP_FILE = 'lookup_distancias_master.csv' 
# Colunas esperadas no arquivo mestre
COLS_MASTER = [
    'Origem - Cidade', 'Destino - Cidade', 
    'Latitude_Origem', 'Longitude_Origem', 
    'Latitude_Destino', 'Longitude_Destino', 
    'Distância (GCD)',
    'Origem_Destino_Key' # Chave auxiliar para busca
]
# Colunas de coordenadas para verificar NaNs
COORD_COLS = ['Latitude_Origem', 'Longitude_Origem', 'Latitude_Destino', 'Longitude_Destino']

# --- Função Auxiliar para Criar Chave ---
def criar_chave_trecho(df):
    # Garante limpeza antes de criar a chave
    origem_col = 'Origem - Cidade'
    destino_col = 'Destino - Cidade'
    if origem_col in df.columns and destino_col in df.columns:
         return df[origem_col].astype(str).str.strip() + '_' + df[destino_col].astype(str).str.strip()
    else:
         raise KeyError(f"Colunas '{origem_col}' ou '{destino_col}' não encontradas para criar chave.")


try:
    # 1. Carregar Lookup Mestre Existente (ou criar vazio)
    print(f"🔄 Tentando carregar lookup mestre: '{MASTER_LOOKUP_FILE}'...")
    if os.path.exists(MASTER_LOOKUP_FILE):
        try:
            df_master_lookup = pd.read_csv(MASTER_LOOKUP_FILE)
            # Verifica se as colunas essenciais existem e adiciona chave
            if all(col in df_master_lookup.columns for col in COLS_MASTER[:-1]): # Verifica todas exceto a chave
                 df_master_lookup['Origem_Destino_Key'] = criar_chave_trecho(df_master_lookup)
                 # Garante tipos numéricos corretos
                 for col in COORD_COLS + ['Distância (GCD)']:
                     if col in df_master_lookup.columns:
                          df_master_lookup[col] = pd.to_numeric(df_master_lookup[col], errors='coerce')
                 print(f"✅ Lookup mestre carregado com {len(df_master_lookup)} trechos.")
            else:
                 print(f"   ⚠️ Arquivo mestre existe mas faltam colunas esperadas. Iniciando com lookup vazio.")
                 df_master_lookup = pd.DataFrame(columns=COLS_MASTER)
        except pd.errors.EmptyDataError:
             print("   ⚠️ Arquivo mestre está vazio. Iniciando com lookup vazio.")
             df_master_lookup = pd.DataFrame(columns=COLS_MASTER)
        except Exception as e_load:
             print(f"   ❌ Erro ao ler arquivo mestre: {e_load}. Iniciando com lookup vazio.")
             df_master_lookup = pd.DataFrame(columns=COLS_MASTER)
    else:
        print("   Arquivo mestre não encontrado. Será criado um novo.")
        df_master_lookup = pd.DataFrame(columns=COLS_MASTER)

    # Garante que a chave existe mesmo se o DF estiver vazio
    if 'Origem_Destino_Key' not in df_master_lookup.columns:
         df_master_lookup['Origem_Destino_Key'] = None

    # 2. Identificar Trechos Únicos Atuais
    if 'trecho_df' not in locals() or trecho_df.empty:
        raise NameError("'trecho_df' filtrado (UFPB Aéreo) não encontrado ou vazio.")
    
    print("🔄 Identificando trechos únicos na execução atual...")
    df_current_trechos = trecho_df[['Origem - Cidade', 'Destino - Cidade']].drop_duplicates().reset_index(drop=True)
    df_current_trechos['Origem_Destino_Key'] = criar_chave_trecho(df_current_trechos)
    print(f"✅ Encontrados {len(df_current_trechos)} trechos únicos na execução atual.")

    # 3. Encontrar Trechos Novos (que não estão no mestre)
    trechos_existentes_keys = set(df_master_lookup['Origem_Destino_Key'].dropna())
    df_new_trechos = df_current_trechos[~df_current_trechos['Origem_Destino_Key'].isin(trechos_existentes_keys)].copy()
    num_new = len(df_new_trechos)
    print(f"🔄 Comparando com lookup mestre: {num_new} trechos novos encontrados para processar.")

    # 4. Processar Apenas os Trechos Novos (Geocoding e Cálculo)
    df_new_calculated_trechos = pd.DataFrame() # Para guardar os resultados dos novos
    if num_new > 0:
        print(f"--- Processando {num_new} Trechos Novos ---")
        # a. Obter lista de cidades únicas APENAS dos novos trechos
        cidades_origem_novas = df_new_trechos['Origem - Cidade'].unique()
        cidades_destino_novas = df_new_trechos['Destino - Cidade'].unique()
        lista_cidades_novas = np.unique(np.concatenate((cidades_origem_novas, cidades_destino_novas))).tolist()
        print(f"   - Identificadas {len(lista_cidades_novas)} cidades únicas novas para geocoding.")

        # b. Obter Coordenadas (Usando a classe Distancia)
        print("   - Obtendo/carregando coordenadas (Geocoding)...")
        # Instancia geocoder aqui para buscar coords apenas das cidades novas
        geocoder = Distancia() 
        df_coordenadas_mapa = geocoder.obter_mapa_coordenadas_hibridas(lista_cidades_novas)
        df_coordenadas_mapa = df_coordenadas_mapa.rename(columns={'Cidade': 'Cidade_Ref', 'Latitude': 'Lat_Ref', 'Longitude': 'Lon_Ref'})

        # c. Juntar Coordenadas aos Trechos Novos
        print("   - Juntando coordenadas aos trechos novos...")
        df_new_trechos = pd.merge(df_new_trechos, df_coordenadas_mapa.add_suffix('_Origem'), left_on='Origem - Cidade', right_on='Cidade_Ref_Origem', how='left')
        df_new_trechos = pd.merge(df_new_trechos, df_coordenadas_mapa.add_suffix('_Destino'), left_on='Destino - Cidade', right_on='Cidade_Ref_Destino', how='left')
        df_new_trechos = df_new_trechos.rename(columns={'Lat_Ref_Origem': 'Latitude_Origem', 'Lon_Ref_Origem': 'Longitude_Origem', 'Lat_Ref_Destino': 'Latitude_Destino', 'Lon_Ref_Destino': 'Longitude_Destino'})
        df_new_trechos = df_new_trechos.drop(columns=['Cidade_Ref_Origem', 'Cidade_Ref_Destino'], errors='ignore')

        # d. Calcular Distância (GCD) para Trechos Novos
        print("   - Calculando distância para trechos novos...")
        # Remove trechos onde faltou coordenada ANTES de calcular
        linhas_antes_drop = len(df_new_trechos)
        df_new_trechos.dropna(subset=COORD_COLS, inplace=True, how='any') 
        num_dropped = linhas_antes_drop - len(df_new_trechos)
        if num_dropped > 0:
             print(f"     ⚠️ {num_dropped} trechos novos removidos por falta de coordenadas.")

        if not df_new_trechos.empty:
            df_new_trechos['Distância (GCD)'] = Distancia.calcular_haversine(
                df_new_trechos['Latitude_Origem'].values, 
                df_new_trechos['Longitude_Origem'].values,
                df_new_trechos['Latitude_Destino'].values,
                df_new_trechos['Longitude_Destino'].values
            )
            # Seleciona e ordena as colunas para o formato mestre
            df_new_calculated_trechos = df_new_trechos[[col for col in COLS_MASTER if col in df_new_trechos.columns]].copy()
            print(f"   ✅ Distância calculada para {len(df_new_calculated_trechos)} trechos novos.")
        else:
            print("   ⚠️ Nenhuma distância calculada para trechos novos (faltaram coordenadas).")
        print("--- Fim do Processamento de Trechos Novos ---")
            
    else: # Caso não haja trechos novos
        print("✅ Nenhum trecho novo para processar.")


    # 5. Atualizar DataFrame Mestre
    if not df_new_calculated_trechos.empty:
        print("🔄 Atualizando lookup mestre com novos trechos...")
        # Garante que df_master_lookup tenha as colunas corretas antes de concatenar
        df_master_lookup = df_master_lookup.reindex(columns=COLS_MASTER) 
        df_master_lookup = pd.concat([df_master_lookup, df_new_calculated_trechos], ignore_index=True)
        # Garante unicidade final (caso algo escape)
        df_master_lookup.drop_duplicates(subset=['Origem_Destino_Key'], keep='last', inplace=True)
        print(f"✅ Lookup mestre atualizado. Total de trechos agora: {len(df_master_lookup)}")
    
    # 6. Salvar Mestre Atualizado
    try:
        df_master_lookup[COLS_MASTER[:-1]].round(6).to_csv(MASTER_LOOKUP_FILE, index=False) # Salva sem a chave auxiliar
        print(f"✅ Lookup mestre atualizado salvo em: '{MASTER_LOOKUP_FILE}'")
    except Exception as e_save:
        print(f"❌ Erro ao salvar o lookup mestre atualizado: {e_save}")


    # 7. Preparar Saída para a Execução Atual ('df_trechos_unicos')
    print("\n🔄 Preparando tabela lookup para a execução atual...")
    # Seleciona do mestre APENAS os trechos que estão na execução ATUAL
    current_keys = set(df_current_trechos['Origem_Destino_Key'])
    df_trechos_unicos = df_master_lookup[df_master_lookup['Origem_Destino_Key'].isin(current_keys)].copy()
    # Remove a chave auxiliar do DataFrame final desta célula
    df_trechos_unicos = df_trechos_unicos.drop(columns=['Origem_Destino_Key'], errors='ignore') 
    
    if not df_trechos_unicos.empty:
        print(f"✅ Tabela 'df_trechos_unicos' pronta para as próximas etapas ({len(df_trechos_unicos)} trechos).")
        print("\n--- Amostra da Tabela de Consulta (para esta execução) ---")
        colunas_amostra = [col for col in COLS_MASTER[:-1] if col in df_trechos_unicos.columns] # Seleciona colunas existentes
        print(df_trechos_unicos[colunas_amostra].head().to_string())
    else:
        print("⚠️ Tabela 'df_trechos_unicos' para esta execução está vazia (nenhum trecho pôde ser processado ou encontrado no mestre).")


except NameError as e:
    print(f"❌ ERRO: Variável {e} não definida. Certifique-se de executar as células anteriores.")
except FileNotFoundError as e:
     print(f"❌ ERRO: Arquivo não encontrado: {e}. Verifique caminhos.")
except KeyError as e:
     print(f"❌ ERRO: Coluna {e} não encontrada. Verifique nomes de colunas.")
except Exception as e:
    print(f"❌ Ocorreu um erro inesperado: {e}")

print("\n--- Fim da Criação/Atualização da Tabela Lookup Mestre ---")


--- Iniciando Criação/Atualização da Tabela Lookup Mestre Persistente ---
🔄 Tentando carregar lookup mestre: 'lookup_distancias_master.csv'...
✅ Lookup mestre carregado com 463 trechos.
🔄 Identificando trechos únicos na execução atual...
✅ Encontrados 172 trechos únicos na execução atual.
🔄 Comparando com lookup mestre: 0 trechos novos encontrados para processar.
✅ Nenhum trecho novo para processar.
✅ Lookup mestre atualizado salvo em: 'lookup_distancias_master.csv'

🔄 Preparando tabela lookup para a execução atual...
✅ Tabela 'df_trechos_unicos' pronta para as próximas etapas (172 trechos).

--- Amostra da Tabela de Consulta (para esta execução) ---
              Origem - Cidade            Destino - Cidade  Latitude_Origem  Longitude_Origem  Latitude_Destino  Longitude_Destino  Distância (GCD)
4                    Campinas        João Pessoa, Paraíba       -22.905822        -47.066350         -7.121598         -34.882028      2186.142790
5        João Pessoa, Paraíba                 

In [148]:
# --- CÉLULA 7: JUNTAR DISTÂNCIAS DA TABELA LOOKUP ---
print("\n--- Juntando Distâncias Pré-calculadas ao trecho_df ---")

try:
    # Verifica se os DFs necessários existem
    if 'trecho_df' not in locals() or trecho_df.empty:
        raise NameError("'trecho_df' (filtrado) não encontrado ou vazio. Execute as células anteriores.")
    if 'df_trechos_unicos' not in locals() or df_trechos_unicos.empty:
        raise NameError("'df_trechos_unicos' (com distâncias) não encontrado ou vazio. Execute a Célula 5.")

    # Prepara a tabela lookup selecionando apenas as colunas chave e a distância
    # Garante que as colunas de origem/destino existam
    if not all(col in df_trechos_unicos.columns for col in [COL_ORIGEM, COL_DESTINO, 'Distância (GCD)']):
        raise KeyError("Colunas 'Origem - Cidade', 'Destino - Cidade' ou 'Distância (GCD)' não encontradas em df_trechos_unicos.")
    
    df_lookup = df_trechos_unicos[[COL_ORIGEM, COL_DESTINO, 'Distância (GCD)']].copy()
    
    # Remove a coluna de distância antiga do trecho_df, se existir
    if 'Distância (GCD)' in trecho_df.columns:
        print("   - Removendo coluna 'Distância (GCD)' existente de trecho_df antes do merge.")
        trecho_df = trecho_df.drop(columns=['Distância (GCD)'])
        
    # Faz o merge para adicionar a distância pré-calculada
    # Garante que as colunas de merge existam em trecho_df
    if not all(col in trecho_df.columns for col in [COL_ORIGEM, COL_DESTINO]):
         raise KeyError(f"Colunas '{COL_ORIGEM}' ou '{COL_DESTINO}' não encontradas em trecho_df para o merge.")
         
    print("   - Realizando merge para adicionar Distância (GCD)...")
    trecho_df = pd.merge(
        trecho_df,
        df_lookup,
        on=[COL_ORIGEM, COL_DESTINO],
        how='left' # Mantém todos os trechos, adiciona NaN se o trecho único não foi calculado
    )
    
    # Verifica quantos trechos ficaram sem distância após o merge
    nulos_pos_merge = trecho_df['Distância (GCD)'].isnull().sum()
    if nulos_pos_merge > 0:
        print(f"⚠️ Atenção: {nulos_pos_merge} trechos não encontraram correspondência na tabela lookup e ficaram sem distância.")
        # Opcional: Listar trechos problemáticos
        # print("   Amostra de trechos sem distância:")
        # print(trecho_df[trecho_df['Distância (GCD)'].isnull()][[COL_ORIGEM, COL_DESTINO]].head())
        
        # Remove trechos que não receberam distância
        trecho_df.dropna(subset=['Distância (GCD)'], inplace=True)
        print(f"   - Trechos sem distância foram removidos. Restantes: {len(trecho_df)}")
    else:
        print("✅ Todas as distâncias foram atribuídas com sucesso a partir da tabela lookup.")
        
    print("\n--- Amostra do trecho_df com distâncias da lookup ---")
    print(trecho_df[[COL_ORIGEM, COL_DESTINO, 'Distância (GCD)']].head().to_string())
    
except NameError as e:
    print(f"❌ ERRO: Variável {e} não definida.")
except KeyError as e:
     print(f"❌ ERRO DE COLUNA: Coluna {e} não encontrada. Verifique os nomes.")
except Exception as e:
     print(f"❌ Ocorreu um erro ao juntar as distâncias: {e}")


--- Juntando Distâncias Pré-calculadas ao trecho_df ---
   - Realizando merge para adicionar Distância (GCD)...
✅ Todas as distâncias foram atribuídas com sucesso a partir da tabela lookup.

--- Amostra do trecho_df com distâncias da lookup ---
              Origem - Cidade            Destino - Cidade  Distância (GCD)
0  Natal, Rio Grande do Norte     Vitória, Espírito Santo      1706.356758
1     Vitória, Espírito Santo  Natal, Rio Grande do Norte      1706.356758
2              Campina Grande                    Campinas      2115.258074
3                    Campinas              Campina Grande      2115.258074
4              Campina Grande                    Campinas      2115.258074


In [149]:
# --- ETAPA SEGUINTE: JUNTAR DISTÂNCIAS DA TABELA LOOKUP ---
print("\n--- Juntando Distâncias Pré-calculadas ao trecho_df ---")

try:
    if 'trecho_df' in locals() and 'df_trechos_unicos' in locals():
        # Prepara a tabela lookup selecionando apenas as colunas chave e a distância
        df_lookup = df_trechos_unicos[[COL_ORIGEM, COL_DESTINO, 'Distância (GCD)']].copy()
        
        # Remove a coluna de distância antiga do trecho_df, se existir de execuções anteriores
        if 'Distância (GCD)' in trecho_df.columns:
            trecho_df = trecho_df.drop(columns=['Distância (GCD)'])
            
        # Faz o merge para adicionar a distância pré-calculada
        trecho_df = pd.merge(
            trecho_df,
            df_lookup,
            on=[COL_ORIGEM, COL_DESTINO],
            how='left' # Mantém todos os trechos, adiciona NaN se o trecho único não foi calculado
        )
        
        # Verifica quantos trechos ficaram sem distância após o merge
        nulos_pos_merge = trecho_df['Distância (GCD)'].isnull().sum()
        if nulos_pos_merge > 0:
            print(f"⚠️ Atenção: {nulos_pos_merge} trechos não encontraram correspondência na tabela lookup e ficaram sem distância.")
            # Remove trechos que não receberam distância
            trecho_df.dropna(subset=['Distância (GCD)'], inplace=True)
            print(f"   - Trechos sem distância foram removidos. Restantes: {len(trecho_df)}")
        else:
            print("✅ Todas as distâncias foram atribuídas com sucesso a partir da tabela lookup.")
            
        print("\n--- Amostra do trecho_df com distâncias da lookup ---")
        print(trecho_df[[COL_ORIGEM, COL_DESTINO, 'Distância (GCD)']].head().to_string())
        
    else:
        print("❌ ERRO: DataFrames 'trecho_df' ou 'df_trechos_unicos' não encontrados.")

except Exception as e:
     print(f"❌ Ocorreu um erro ao juntar as distâncias: {e}")


--- Juntando Distâncias Pré-calculadas ao trecho_df ---
✅ Todas as distâncias foram atribuídas com sucesso a partir da tabela lookup.

--- Amostra do trecho_df com distâncias da lookup ---
              Origem - Cidade            Destino - Cidade  Distância (GCD)
0  Natal, Rio Grande do Norte     Vitória, Espírito Santo      1706.356758
1     Vitória, Espírito Santo  Natal, Rio Grande do Norte      1706.356758
2              Campina Grande                    Campinas      2115.258074
3                    Campinas              Campina Grande      2115.258074
4              Campina Grande                    Campinas      2115.258074


## itinerarios 

In [150]:
# --- ETAPA 8 (CORRIGIDA v2): CRIAR/ATUALIZAR ARQUIVO MESTRE DE ITINERÁRIOS ---
import pandas as pd
import numpy as np
import json
import os
from tqdm.auto import tqdm

tqdm.pandas(desc="Processando Novos Itinerários")

print("\n--- Iniciando Criação/Atualização do Arquivo Mestre de Itinerários ---")

# --- Constantes ---
MASTER_ITINERARIOS_FILE = 'itinerarios_master.jsonl'
COL_ID_VIAGEM = 'Identificador do processo de viagem'
# Nomes das colunas de coordenadas ESPERADAS em df_trechos_unicos (da Célula 5)
COORD_COLS = ['Latitude_Origem', 'Longitude_Origem', 'Latitude_Destino', 'Longitude_Destino']

# --- Função para formatar (Agora espera coordenadas) ---
def formatar_itinerario(group):
    distancia_total = group['Distância (GCD)'].sum() # Coluna da Célula 7
    trechos_lista = []
    for _, row in group.iterrows():
        # Acessa diretamente - se a coluna faltar, dará erro (bom para debug)
        trecho_info = {
            'Origem': row[COL_ORIGEM],
            'Destino': row[COL_DESTINO],
            'Lat_Origem': row['Latitude_Origem'], 
            'Lon_Origem': row['Longitude_Origem'],
            'Lat_Destino': row['Latitude_Destino'],
            'Lon_Destino': row['Longitude_Destino'],
            'Distancia_Trecho': row['Distância (GCD)']
        }
        trecho_info = {k: round(v, 6) if isinstance(v, (int, float)) and pd.notna(v) else v for k, v in trecho_info.items()}
        trechos_lista.append(trecho_info)

    return pd.Series({
        'Distancia_Total_Viagem': round(distancia_total, 4),
        'Itinerario_Detalhado': trechos_lista
    })

try:
    # 1. Carregar Itinerários Mestres Existentes
    print(f"🔄 Tentando carregar arquivo mestre de itinerários: '{MASTER_ITINERARIOS_FILE}'...")
    master_itineraries_dict = {}
    # ... (código para carregar o mestre - sem alterações) ...
    if os.path.exists(MASTER_ITINERARIOS_FILE):
        try:
            with open(MASTER_ITINERARIOS_FILE, 'r', encoding='utf-8') as f:
                # ... (loop para ler linhas json) ...
                 for line in f:
                    try:
                        data = json.loads(line)
                        if COL_ID_VIAGEM in data and 'Distancia_Total_Viagem' in data and 'Itinerario_Detalhado' in data:
                             trip_id = str(data[COL_ID_VIAGEM]).replace('.0','').strip().zfill(21)
                             master_itineraries_dict[trip_id] = {
                                 'Distancia_Total_Viagem': data['Distancia_Total_Viagem'],
                                 'Itinerario_Detalhado': data['Itinerario_Detalhado']
                             }
                    except json.JSONDecodeError:
                        print(f"   ⚠️ Erro ao decodificar linha JSON ignorada: {line.strip()}")
            print(f"✅ Arquivo mestre carregado com {len(master_itineraries_dict)} itinerários.")
        except Exception as e_load:
            print(f"   ❌ Erro ao ler arquivo mestre: {e_load}. Iniciando com dicionário vazio.")
            master_itineraries_dict = {}
    else:
        print("   Arquivo mestre não encontrado. Será criado um novo.")
        master_itineraries_dict = {}


    # 2. Preparar Dados Atuais (trecho_df + Coordenadas)
    if 'trecho_df' not in locals() or trecho_df.empty:
        raise NameError("'trecho_df' (com distâncias da Célula 7) não encontrado ou vazio.")
    if 'df_trechos_unicos' not in locals() or df_trechos_unicos.empty:
         raise NameError("'df_trechos_unicos' (com coordenadas da Célula 5) não encontrado ou vazio.")

    print("🔄 Preparando trecho_df com coordenadas para processamento...")

    # Verifica se df_trechos_unicos tem as colunas necessárias
    colunas_lookup_necessarias = [COL_ORIGEM, COL_DESTINO] + COORD_COLS
    if not all(col in df_trechos_unicos.columns for col in colunas_lookup_necessarias):
         col_faltante_lookup = next((col for col in colunas_lookup_necessarias if col not in df_trechos_unicos.columns), 'Desconhecida')
         raise KeyError(f"Coluna essencial '{col_faltante_lookup}' não encontrada em df_trechos_unicos (da Célula 5).")

    # Seleciona apenas as colunas necessárias da lookup para o merge
    df_coords_lookup = df_trechos_unicos[colunas_lookup_necessarias].drop_duplicates(subset=[COL_ORIGEM, COL_DESTINO])

    # Remove colunas de coordenadas antigas do trecho_df (resultado da Célula 7), se existirem
    trecho_df_base = trecho_df.drop(columns=[col for col in COORD_COLS if col in trecho_df.columns], errors='ignore')

    # Junta as coordenadas corretas ao trecho_df que já tem a Distância (GCD)
    trecho_df_completo_para_grupo = pd.merge(
        trecho_df_base, # Usa o DF base sem as coords antigas
        df_coords_lookup,
        on=[COL_ORIGEM, COL_DESTINO], # Chave de merge
        how='left' # Mantém todos os trechos
    )
    
    # --- DEBUGGING: MOSTRAR COLUNAS APÓS MERGE ---
    print("   - Colunas em 'trecho_df_completo_para_grupo' após merge:", trecho_df_completo_para_grupo.columns.tolist())
    # --------------------------------------------

    # Verifica se o merge adicionou as colunas e se há nulos
    if not all(col in trecho_df_completo_para_grupo.columns for col in COORD_COLS):
         print("   ❌ ERRO PÓS-MERGE: Colunas de coordenadas não foram adicionadas corretamente.")
         # Identifica qual coluna faltou
         col_faltante_pos_merge = next((col for col in COORD_COLS if col not in trecho_df_completo_para_grupo.columns), 'Desconhecida')
         raise KeyError(f"Coluna '{col_faltante_pos_merge}' ausente após merge.")
         
    nulos_coords = trecho_df_completo_para_grupo[COORD_COLS].isnull().sum()
    if nulos_coords.any():
        print(f"   ⚠️ Atenção: {nulos_coords.sum()} valores de coordenadas ausentes após merge. Verifique df_trechos_unicos.")
        # Decide como tratar: remover ou permitir NaNs na saída
        # trecho_df_completo_para_grupo.dropna(subset=COORD_COLS, inplace=True, how='any')
        # print("      -> Trechos com coordenadas ausentes foram removidos.")


    # Garante padronização do ID
    trecho_df_completo_para_grupo[COL_ID_VIAGEM] = trecho_df_completo_para_grupo[COL_ID_VIAGEM].astype(str).str.strip().str.zfill(21)

    # Ordena
    coluna_sequencia = 'Sequência Trecho'
    if coluna_sequencia in trecho_df_completo_para_grupo.columns:
        trecho_df_completo_para_grupo[coluna_sequencia] = pd.to_numeric(trecho_df_completo_para_grupo[coluna_sequencia], errors='coerce').fillna(999)
        trecho_df_completo_para_grupo.sort_values(by=[COL_ID_VIAGEM, coluna_sequencia], inplace=True)
        print("   - Trechos ordenados por ID e Sequência.")
    else:
        trecho_df_completo_para_grupo.sort_values(by=COL_ID_VIAGEM, inplace=True)
        print(f"   ⚠️ Atenção: Coluna '{coluna_sequencia}' não encontrada. Trechos ordenados apenas por ID.")

    # 3. Identificar IDs Novos
    current_ids = set(trecho_df_completo_para_grupo[COL_ID_VIAGEM].unique())
    master_ids = set(master_itineraries_dict.keys())
    new_ids = current_ids - master_ids
    num_new = len(new_ids)
    print(f"🔄 Comparando com arquivo mestre: {num_new} IDs de viagem novos encontrados para processar.")

    # 4. Processar Apenas os Itinerários Novos
    df_novos_itinerarios = pd.DataFrame()
    if num_new > 0:
        print(f"--- Processando {num_new} Itinerários Novos ---")
        trecho_df_novos = trecho_df_completo_para_grupo[trecho_df_completo_para_grupo[COL_ID_VIAGEM].isin(new_ids)].copy()

        if not trecho_df_novos.empty:
            # Verifica se as colunas de coordenadas existem ANTES de aplicar
            if not all(col in trecho_df_novos.columns for col in COORD_COLS):
                 print("   ❌ ERRO CRÍTICO: Colunas de coordenadas ausentes no DataFrame de novos trechos antes do apply.")
            else:
                 df_novos_itinerarios = trecho_df_novos.groupby(COL_ID_VIAGEM).progress_apply(formatar_itinerario).reset_index()
                 print(f"\n   ✅ {len(df_novos_itinerarios)} itinerários novos formatados.")
        else:
            print("   ⚠️ DataFrame filtrado para novos IDs está vazio.")

        # 5. Adicionar Novos ao Mestre
        if not df_novos_itinerarios.empty:
            print("   🔄 Atualizando arquivo mestre com novos itinerários...")
            # ... (código para adicionar ao mestre - sem alterações) ...
            try:
                with open(MASTER_ITINERARIOS_FILE, 'a', encoding='utf-8') as f:
                     for _, row in df_novos_itinerarios.iterrows():
                        record = { COL_ID_VIAGEM: row[COL_ID_VIAGEM], 'Distancia_Total_Viagem': row['Distancia_Total_Viagem'], 'Itinerario_Detalhado': row['Itinerario_Detalhado'] }
                        json.dump(record, f, ensure_ascii=False); f.write('\n')
                        master_itineraries_dict[row[COL_ID_VIAGEM]] = { 'Distancia_Total_Viagem': row['Distancia_Total_Viagem'], 'Itinerario_Detalhado': row['Itinerario_Detalhado'] }
                print(f"   ✅ Novos itinerários adicionados a '{MASTER_ITINERARIOS_FILE}'.")
            except Exception as e_save: print(f"   ❌ Erro ao salvar novos itinerários no arquivo mestre: {e_save}")
            
    else:
        print("✅ Nenhum itinerário novo para processar.")

    # 6. Preparar Saída para a Execução Atual ('df_itinerarios')
    print("\n🔄 Preparando DataFrame de itinerários para a execução atual...")
    # ... (código para preparar saída - sem alterações) ...
    current_itineraries_data = []
    for trip_id in current_ids:
        if trip_id in master_itineraries_dict:
            data = master_itineraries_dict[trip_id]; data[COL_ID_VIAGEM] = trip_id
            current_itineraries_data.append(data)

    if current_itineraries_data:
         df_itinerarios = pd.DataFrame(current_itineraries_data)[[COL_ID_VIAGEM, 'Distancia_Total_Viagem', 'Itinerario_Detalhado']]
         print(f"✅ DataFrame 'df_itinerarios' pronto ({len(df_itinerarios)} itinerários).")

        #  # 7. Salvar Saídas Opcionais
        #  # ... (código para salvar CSV/JSON da execução - sem alterações) ...
        #  try:
        #      ano = 2023 # Garante que 'ano' está definido
        #      pasta_saida = f'dadosViagens/dados_viagens{ano}' 
        #      if not os.path.exists(pasta_saida): os.makedirs(pasta_saida)
        #      arquivo_itinerarios_csv_run = os.path.join(pasta_saida, f'itinerarios_detalhados_ufpb_aereo_{ano}.csv')
        #      df_itinerarios.to_csv(arquivo_itinerarios_csv_run, index=False)
        #      arquivo_itinerarios_json_run = os.path.join(pasta_saida, f'itinerarios_detalhados_ufpb_aereo_{ano}.json')
        #      df_itinerarios.to_json(arquivo_itinerarios_json_run, orient='records', indent=4)
        #  except Exception as e_save_run: print(f"   -> Erro ao salvar arquivos opcionais: {e_save_run}")


         print("\n--- Amostra do DataFrame de Itinerários (para esta execução) ---")
         with pd.option_context('display.max_colwidth', None):
             print(df_itinerarios.head().to_string())
    else:
         print("⚠️ DataFrame 'df_itinerarios' para esta execução está vazio.")
         df_itinerarios = pd.DataFrame(columns=[COL_ID_VIAGEM, 'Distancia_Total_Viagem', 'Itinerario_Detalhado'])


except NameError as e:
    print(f"❌ ERRO: Variável {e} não definida. Certifique-se de executar as células anteriores (Célula 5 e Célula 7).")
except KeyError as e:
     print(f"❌ ERRO DE COLUNA: Coluna {e} não encontrada. Verifique se a célula anterior (Célula 7) adicionou 'Distância (GCD)' ao 'trecho_df' ou se as colunas de coordenadas existem em 'df_trechos_unicos'.")
except Exception as e:
    print(f"❌ Ocorreu um erro inesperado ao estruturar itinerários: {e}")

print("\n--- Fim da Criação/Atualização do Arquivo Mestre de Itinerários ---")


--- Iniciando Criação/Atualização do Arquivo Mestre de Itinerários ---
🔄 Tentando carregar arquivo mestre de itinerários: 'itinerarios_master.jsonl'...
✅ Arquivo mestre carregado com 1893 itinerários.
🔄 Preparando trecho_df com coordenadas para processamento...
   - Colunas em 'trecho_df_completo_para_grupo' após merge: ['Identificador do processo de viagem', 'Número da Proposta (PCDP)', 'Sequência Trecho', 'Origem - Data', 'Origem - País', 'Origem - UF', 'Origem - Cidade', 'Destino - Data', 'Destino - País', 'Destino - UF', 'Destino - Cidade', 'Meio de transporte', 'Número Diárias', 'Missao?', 'Distância (GCD)', 'Latitude_Origem', 'Longitude_Origem', 'Latitude_Destino', 'Longitude_Destino']
   - Trechos ordenados por ID e Sequência.
🔄 Comparando com arquivo mestre: 0 IDs de viagem novos encontrados para processar.
✅ Nenhum itinerário novo para processar.

🔄 Preparando DataFrame de itinerários para a execução atual...
✅ DataFrame 'df_itinerarios' pronto (320 itinerários).

--- Amostra

In [151]:
import os
import json 


# Carrega os arquivos lookup e itinerários
try:
    lookup = pd.read_csv(f"lookup_distancias_master.csv")
    pasta_saida = f'dadosViagens/dados_viagens{ano}' # Garante que pasta_saida está definida
    
    # --- CORREÇÃO AQUI ---
    itinerarios_path = f'itinerarios_master.jsonl'
    itinerarios = pd.read_json(itinerarios_path, lines=True) 
    # ---------------------
    
    print("✅ Arquivos lookup_distancias_master.csv e itinerarios_master.jsonl carregados.")

    # Criar dicionário de referência (origem, destino) → distância
    dist_ref = {
        # Adiciona strip() para segurança, caso haja espaços extras no CSV
        (str(row["Origem - Cidade"]).strip(), str(row["Destino - Cidade"]).strip()): row["Distância (GCD)"]
        for _, row in lookup.iterrows()
    }
    print(f"✅ Dicionário de referência de distâncias criado com {len(dist_ref)} entradas.")
    
    # Exibe amostra dos dados carregados (opcional)
    print("\nAmostra do lookup:")
    print(lookup.head())
    print("\nAmostra dos itinerários:")
    print(itinerarios.head())


except FileNotFoundError as e:
     print(f"❌ ERRO: Arquivo não encontrado: {e}. Verifique se os arquivos estão na mesma pasta do notebook.")
except ValueError as e:
     print(f"❌ ERRO ao ler JSON: {e}. Verifique se o arquivo '{itinerarios_path}' está no formato JSON Lines correto.")
except Exception as e:
     print(f"❌ Ocorreu um erro inesperado: {e}")

✅ Arquivos lookup_distancias_master.csv e itinerarios_master.jsonl carregados.
✅ Dicionário de referência de distâncias criado com 463 entradas.

Amostra do lookup:
        Origem - Cidade      Destino - Cidade  Latitude_Origem  \
0                Tóquio  João Pessoa, Paraíba        35.676422   
1  João Pessoa, Paraíba                Tóquio        -7.121598   
2             Las Vegas  João Pessoa, Paraíba        36.171563   
3  João Pessoa, Paraíba             Las Vegas        -7.121598   
4              Campinas  João Pessoa, Paraíba       -22.905822   

   Longitude_Origem  Latitude_Destino  Longitude_Destino  Distância (GCD)  
0        139.650027         -7.121598         -34.882028     16791.388079  
1        -34.882028         35.676422         139.650027     16791.388079  
2       -115.139101         -7.121598         -34.882028      9609.831314  
3        -34.882028         36.171563        -115.139101      9609.831314  
4        -47.066350         -7.121598         -34.882028  

In [152]:
# import numpy as np
# import pandas as pd

# # Flag para controlar se a agregação foi bem-sucedida
# agregacao_ok = False
# df_viagens_agregadas = pd.DataFrame() # Inicializa como DataFrame vazio

# # --- ETAPA 1: AGREGAÇÃO DA DISTÂNCIA TOTAL POR VIAGEM ---
# print("🔄 Iniciando agregação de trechos por viagem...")

# # Garante que o trecho_df exista antes de agregar
# try:
#     # Agrupa o trecho_df por viagem e soma as distâncias
#     df_viagens_agregadas = (
#         trecho_df.groupby("Identificador do processo de viagem", as_index=False)
#         .agg({
#             "Distância (GCD)": "sum",          # SOMA a distância total
#             "Origem - Cidade": "first",        # Pega a primeira origem da viagem
#             "Destino - Cidade": "last",         # Pega o último destino da viagem
#         })
#     )
#     if not df_viagens_agregadas.empty:
#         print(f"✅ Agregação concluída. DataFrame 'df_viagens_agregadas' criado com {len(df_viagens_agregadas)} viagens únicas.")
#         agregacao_ok = True
#     else:
#         print("⚠️ Agregação resultou em DataFrame vazio.")

# except NameError:
#     print("❌ ERRO CRÍTICO: O DataFrame 'trecho_df' não foi encontrado. Certifique-se de executar as células anteriores (cálculo de distância) primeiro.")
# except Exception as e:
#     print(f"❌ Ocorreu um erro inesperado durante a agregação: {e}")


# # --- ETAPA 2 (CORRIGIDA): CÁLCULO DE EMISSÕES (SEM UPLIFT, APENAS RFI) ---
# # Só executa se a agregação (Etapa 1) foi bem-sucedida
# if agregacao_ok:
#     print("\n🔄 [TESTE FINAL] Calculando emissões SEM Uplift (apenas RFI), para alinhamento com baseline...")


#     # --- FATORES DE EMISSÃO (EF) em kgCO2eq/km ---
#     # <<< VERIFIQUE SE ESTES VALORES ESTÃO CORRETOS CONFORME SUA DOCUMENTAÇÃO >>>
#     EF_MUITO_CURTA_EVITAVEL = 0.272576785
#     EF_CURTA_DISTANCIA      = 0.182869354
#     EF_LONGA_DISTANCIA      = 0.200108215
#     # <<< FIM DA VERIFICAÇÃO >>>

#     # Limites de Distância (em km)
#     LIMITE_DH = 600
#     LIMITE_SH = 3700

#     conditions = [
#         df_viagens_agregadas['Distância (GCD)'].le(LIMITE_DH),
#         (df_viagens_agregadas['Distância (GCD)'].gt(LIMITE_DH)) & (df_viagens_agregadas['Distância (GCD)'].le(LIMITE_SH)),
#         df_viagens_agregadas['Distância (GCD)'].gt(LIMITE_SH)
#     ]

#     category_values = ['Muito Curta (Evitável)', 'Curta Distância', 'Longa Distância']
#     ef_values = [EF_MUITO_CURTA_EVITAVEL, EF_CURTA_DISTANCIA, EF_LONGA_DISTANCIA]

#     df_viagens_agregadas['Categoria Distância'] = np.select(conditions, category_values, default='Não Calculada')
#     df_viagens_agregadas['EF Aplicado'] = np.select(conditions, ef_values, default=np.nan)

    
#     # Fórmula (SEM Uplift)
#     df_viagens_agregadas['Emissões (KgCO2eq)'] = (df_viagens_agregadas['Distância (GCD)']) * df_viagens_agregadas['EF Aplicado'] 

#     print(f"✅ Emissões calculadas.")

#     print("\n--- Amostra do cálculo corrigido ---")
#     print(df_viagens_agregadas[['Identificador do processo de viagem', 'Distância (GCD)', 'Categoria Distância', 'Emissões (KgCO2eq)']].head().to_string())
# else:
#     print("\n⚠️ O cálculo de emissões foi pulado porque o DataFrame 'df_viagens_agregadas' não foi criado com sucesso na Etapa 1.")

# Etapa 2: Agregação por Viagem e Cálculo de Emissões (LÓGICA CORRIGIDA)

In [153]:
# --- ETAPA 9 (REVISADA): AGREGAÇÃO (COM CONTAGEM DE TRECHOS) E CÁLCULO DE EMISSÕES ---
import numpy as np
import pandas as pd

# Flag para controlar se a agregação foi bem-sucedida
agregacao_ok = False
df_viagens_agregadas = pd.DataFrame() # Inicializa como DataFrame vazio

print("🔄 Iniciando agregação dos trechos filtrados (UFPB Aéreo) por viagem...")

# Garante que o trecho_df (já com distâncias da lookup) exista
try:
    if 'trecho_df' not in locals() or trecho_df.empty:
         raise NameError("'trecho_df' (com distâncias da lookup) não encontrado ou vazio.")

    # Agrupa o trecho_df filtrado por viagem
    df_viagens_agregadas = (
        trecho_df.groupby("Identificador do processo de viagem", as_index=False)
        .agg(
            # Renomeia colunas para o formato pd.NamedAgg
            **{'Distância (GCD)': pd.NamedAgg(column='Distância (GCD)', aggfunc='sum'),
               'Origem - Cidade': pd.NamedAgg(column='Origem - Cidade', aggfunc='first'),
               'Destino - Cidade': pd.NamedAgg(column='Destino - Cidade', aggfunc='last'),
               # --- NOVA LINHA ADICIONADA ---
               # Conta quantos trechos (linhas) existem para este ID de viagem
               'Total_Trechos': pd.NamedAgg(column='Origem - Cidade', aggfunc='count') 
               # --- FIM DA NOVA LINHA ---
              }
        )
    )
    if not df_viagens_agregadas.empty:
        print(f"✅ Agregação concluída. DataFrame 'df_viagens_agregadas' criado com {len(df_viagens_agregadas)} viagens únicas (UFPB Aéreo).")
        agregacao_ok = True
    else:
        print("⚠️ Agregação resultou em DataFrame vazio.")

except NameError as e:
    print(f"❌ ERRO CRÍTICO: {e}. Certifique-se de executar as células anteriores.")
except Exception as e:
    print(f"❌ Ocorreu um erro inesperado durante a agregação: {e}")


# --- CÁLCULO DE EMISSÕES (SIMPLIFICADO: USANDO EF QUE JÁ INCLUI RFI/UPLIFT) ---
# Só executa se a agregação foi bem-sucedida
if agregacao_ok:
    print("\n🔄 Calculando emissões usando Fatores de Emissão (EF) que já incluem RFI/Uplift...")

    # --- FATORES DE EMISSÃO (EF) em kg CO2e / pass.km ---
    # <<< ATENÇÃO: CONFIRME SE ESTES SÃO OS VALORES EF CORRETOS DO SEU CONSTANTES.CSV / IMAGEM >>>
    EF_ECO_MUITO_CURTA = 0.272576785 
    EF_ECO_CURTA       = 0.182869354 
    EF_ECO_LONGA       = 0.200108215
    # <<< FIM DA CONFIRMAÇÃO >>>

    LIMITE_DH = 600
    LIMITE_SH = 3700

    # Categorias baseadas na Distância GCD PURA
    conditions = [
        df_viagens_agregadas['Distância (GCD)'].le(LIMITE_DH),
        (df_viagens_agregadas['Distância (GCD)'].gt(LIMITE_DH)) & (df_viagens_agregadas['Distância (GCD)'].le(LIMITE_SH)),
        df_viagens_agregadas['Distância (GCD)'].gt(LIMITE_SH)
    ]

    category_values = ['Muito Curta (Evitável)', 'Curta Distância', 'Longa Distância']
    # Use os valores EF (CO2e) corretos aqui
    ef_values = [EF_ECO_MUITO_CURTA, EF_ECO_CURTA, EF_ECO_LONGA]

    df_viagens_agregadas['Categoria Distância'] = np.select(conditions, category_values, default='Não Calculada')
    df_viagens_agregadas['EF Aplicado'] = np.select(conditions, ef_values, default=np.nan)

    # --- APLICAÇÃO DA FÓRMULA SIMPLIFICADA ---
    # Emissões = Distância_GCD * EF_CO2e (que já inclui RFI/Uplift)
    df_viagens_agregadas['Emissões (KgCO2eq)'] = df_viagens_agregadas['Distância (GCD)'] * df_viagens_agregadas['EF Aplicado']

    # Arredondamento final das emissões
    df_viagens_agregadas['Emissões (KgCO2eq)'] = df_viagens_agregadas['Emissões (KgCO2eq)'].round(4)


    print("✅ Emissões calculadas usando EF pré-ajustados (sem multiplicar RFI/Uplift novamente).")

    print("\n--- Amostra do cálculo simplificado ---")
    # --- PRINT ATUALIZADO PARA INCLUIR Total_Trechos ---
    colunas_amostra_agg = ['Identificador do processo de viagem', 'Distância (GCD)', 'Total_Trechos', 'Categoria Distância', 'Emissões (KgCO2eq)']
    print(df_viagens_agregadas[colunas_amostra_agg].head().to_string())
    # --- FIM DA ATUALIZAÇÃO DO PRINT ---
else:
    print("\n⚠️ O cálculo de emissões foi pulado porque o DataFrame 'df_viagens_agregadas' não foi criado com sucesso na etapa anterior.")

🔄 Iniciando agregação dos trechos filtrados (UFPB Aéreo) por viagem...
✅ Agregação concluída. DataFrame 'df_viagens_agregadas' criado com 320 viagens únicas (UFPB Aéreo).

🔄 Calculando emissões usando Fatores de Emissão (EF) que já incluem RFI/Uplift...
✅ Emissões calculadas usando EF pré-ajustados (sem multiplicar RFI/Uplift novamente).

--- Amostra do cálculo simplificado ---
  Identificador do processo de viagem  Distância (GCD)  Total_Trechos Categoria Distância  Emissões (KgCO2eq)
0               000000000000018548551      3412.713516              2     Curta Distância            624.0807
1               000000000000018559255      4230.516148              2     Longa Distância            846.5610
2               000000000000018559668      4230.516148              2     Longa Distância            846.5610
3               000000000000018657968      3424.980642              2     Curta Distância            626.3240
4               000000000000018666746     11373.956459              3

# Etapa 3: Merge Final e Criação do DataFrame Mestre

In [154]:
# --- ETAPA DE MERGE FINAL (CRIANDO df_master_ufpb_aereo) ---
print("\n--- Iniciando Merge Final para criar DataFrame Mestre (UFPB Aéreo) ---")

try:
    # Verifica se os DataFrames necessários existem
    if 'df_viagens_agregadas' not in locals() or df_viagens_agregadas.empty:
         raise NameError("'df_viagens_agregadas' (UFPB Aéreo agregado) não encontrado ou vazio.")
    if 'viagem_df' not in locals() or 'passagem_df' not in locals():
        raise NameError("DataFrames originais 'viagem_df' ou 'passagem_df' não encontrados.")

    # 1. Preparar DataFrames Originais para Merge
    #    - Garantir que IDs sejam strings e limpos
    #    - Selecionar colunas úteis e remover duplicatas por ID de viagem
    
    viagem_df[COL_ID_TRECHO] = viagem_df[COL_ID_TRECHO].astype(str).str.strip()
    viagem_df_unique = viagem_df.drop_duplicates(subset=[COL_ID_TRECHO], keep='first')
    # Selecione colunas relevantes do viagem_df (exceto as que já estão em df_viagens_agregadas)
    colunas_viagem = [col for col in viagem_df_unique.columns if col not in ['Origem - Cidade', 'Destino - Cidade', 'Distância (GCD)', 'Categoria Distância', 'Emissões (KgCO2eq)', 'EF Aplicado']] 
    
    passagem_df[COL_ID_TRECHO] = passagem_df[COL_ID_TRECHO].astype(str).str.strip()
    passagem_df_unique = passagem_df.drop_duplicates(subset=[COL_ID_TRECHO], keep='first')
     # Selecione colunas relevantes do passagem_df 
    colunas_passagem = [col for col in passagem_df_unique.columns if col != COL_ID_TRECHO] # Exclui ID para evitar sufixo

    # 2. Realizar os Merges
    print("🔄 Juntando dados de viagem...")
    df_master_ufpb_aereo = pd.merge(
        df_viagens_agregadas,
        viagem_df_unique[colunas_viagem],
        on=COL_ID_TRECHO,
        how='left' # Mantém todas as viagens UFPB agregadas
    )

    print("🔄 Juntando dados de passagem...")
    df_master_ufpb_aereo = pd.merge(
        df_master_ufpb_aereo,
        passagem_df_unique[colunas_passagem + [COL_ID_TRECHO]], # Inclui ID para merge
        on=COL_ID_TRECHO,
        how='left',
        suffixes=('', '_PASSAGEM') # Adiciona sufixo caso haja colunas com mesmo nome
    )

    # 3. Limpeza Final (Remover colunas duplicadas ou desnecessárias)
    colunas_redundantes = [col for col in df_master_ufpb_aereo.columns if col.endswith('_PASSAGEM')]
    df_master_ufpb_aereo = df_master_ufpb_aereo.drop(columns=colunas_redundantes, errors='ignore')
    
    # Adicione outras limpezas se necessário (ex: renomear colunas)

    print(f"✅ Merge final concluído. DataFrame 'df_master_ufpb_aereo' criado com {len(df_master_ufpb_aereo)} linhas.")

    print("\n--- Amostra do DataFrame Mestre Final (UFPB Aéreo) ---")
    print(df_master_ufpb_aereo.head().to_string()) # Mostra mais colunas

except NameError as e:
    print(f"❌ ERRO CRÍTICO: {e}. Certifique-se de executar as células anteriores.")
except KeyError as e:
     print(f"❌ ERRO: Coluna {e} não encontrada durante o merge. Verifique os nomes das colunas nos DataFrames originais.")
except Exception as e:
    print(f"❌ Ocorreu um erro inesperado durante o merge final: {e}")
    
    

# ... (após os merges) ...
    print(f"✅ Merge final concluído. DataFrame 'df_master_ufpb_aereo' criado com {len(df_master_ufpb_aereo)} linhas.")

    # --- ADICIONAR ARREDONDAMENTO FINAL ANTES DE SALVAR ---
    print("🔄 Arredondando Distância e Emissões finais para inteiros...")
    if 'Distância (GCD)' in df_master_ufpb_aereo.columns:
         df_master_ufpb_aereo['Distância (GCD)'] = df_master_ufpb_aereo['Distância (GCD)'].round().astype(int)
    if 'Emissões (KgCO2eq)' in df_master_ufpb_aereo.columns:
         df_master_ufpb_aereo['Emissões (KgCO2eq)'] = df_master_ufpb_aereo['Emissões (KgCO2eq)'].round().astype(int)
    # ---------------------------------------------------

    print("\n--- Amostra do DataFrame Mestre Final (UFPB Aéreo) ---")
    print(df_master_ufpb_aereo.head().to_string())
    



--- Iniciando Merge Final para criar DataFrame Mestre (UFPB Aéreo) ---
🔄 Juntando dados de viagem...
🔄 Juntando dados de passagem...
✅ Merge final concluído. DataFrame 'df_master_ufpb_aereo' criado com 320 linhas.

--- Amostra do DataFrame Mestre Final (UFPB Aéreo) ---
  Identificador do processo de viagem  Distância (GCD)             Origem - Cidade            Destino - Cidade  Total_Trechos Categoria Distância  EF Aplicado  Emissões (KgCO2eq) Número da Proposta (PCDP) Meio de transporte País - Origem ida      UF - Origem ida Cidade - Origem ida País - Destino ida     UF - Destino ida Cidade - Destino ida País - Origem volta UF - Origem volta Cidade - Origem volta Pais - Destino volta UF - Destino volta Cidade - Destino volta Valor da passagem Taxa de serviço Data da emissão/compra Hora da emissão/compra   Situação Viagem Urgente                                                                                                       Justificativa Urgência Viagem Código do órgão superior

# Etapa 4: Salvar DataFrame Mestre Completo

In [155]:
# --- ETAPA PARA SALVAR O RESULTADO FINAL ---
print("\n--- Salvando DataFrame Mestre Final (UFPB Aéreo) ---")

try:
    if 'df_master_ufpb_aereo' in locals() and not df_master_ufpb_aereo.empty:
        # <<< MUDANÇA AQUI >>>
        arquivo_final = ARQUIVO_MASTER_OUT # Usa variável da Célula de Config
        df_master_ufpb_aereo.to_csv(arquivo_final, index=False)
        print(f"✅ DataFrame final ({ORGAO_NOME_CURTO}) salvo com sucesso em: '{arquivo_final}'")
    
    else:
        print("⚠️ O DataFrame 'df_master_ufpb_aereo' está vazio ou não foi definido. Nada foi salvo.")

except Exception as e:
    print(f"❌ Ocorreu um erro ao salvar o arquivo final: {e}")


--- Salvando DataFrame Mestre Final (UFPB Aéreo) ---
✅ DataFrame final (UFCG) salvo com sucesso em: 'dadosViagens/dados_viagens2025\df_master_UFCG_aereo_2025.csv'


In [156]:
# # carregar o arquivo salvo para verificação
# import os
# import pandas as pd

# ano = 2023
# pasta_saida = f'dadosViagens/dados_viagens{ano}'
# if not os.path.exists(pasta_saida):
#     os.makedirs(pasta_saida)
            
# arquivo_final = os.path.join(pasta_saida, f'df_master_ufpb_aereo_{ano}.csv')
# df_master_ufpb_aereo = pd.read_csv(arquivo_final)

In [157]:
# soma = df_master_ufpb_aereo["Distância (GCD)"].sum()
# print(f"Soma total das distâncias no df_master_ufpb_aereo: {soma} km")

# Etapa 6: Comparar com Baseline e Gerar `diferenca_corrigida.csv`

In [158]:
# # --- ETAPA 6 (REVISADA): COMPARAR COM BASELINE DE DOIS ARQUIVOS (2023.1 e 2023.2) ---
# import pandas as pd
# import numpy as np
# import os
# import glob # Usado para encontrar o arquivo do notebook

# print("\n--- Iniciando Comparação com Baseline Dividido (2023.1 e 2023.2) ---")

# # --- VARIÁVEIS DE CONFIGURAÇÃO ---
# ano = 2023 # Ano principal
# pasta_resultados_notebook = f'dadosViagens/dados_viagens{ano}'
# df_ufpb_path = os.path.join(pasta_resultados_notebook, f'df_master_ufpb_aereo_{ano}.csv')
# arquivo_itinerarios = os.path.join(pasta_resultados_notebook, f'itinerarios_detalhados_ufpb_aereo_{ano}.csv') # Opcional

# # --- CAMINHOS DOS ARQUIVOS BASELINE ---
# # Caminho para o diretório dos arquivos de baseline (ajuste se necessário)
# baseline_dir = r"C:\Users\augusto\Documents\Portal da Transparência\2023_20240512_Viagens"

# # Arquivo Baseline 2023.2 (Semestre 2)
# local_baseline_2023_2 = os.path.join(baseline_dir, "Modelo 2023.2 Transparência - English.xlsx")
# sheet_name_baseline_2023_2 = 'Entrada Lvl 5-4 qt'

# # Arquivo Baseline 2023.1 (Semestre 1)
# local_baseline_2023_1 = os.path.join(baseline_dir, "Modelo 2023.1 Transparência.xlsx") # Nome do arquivo 2023.1
# sheet_name_baseline_2023_1 = 'Entrada Lvl 5-4 qt' # Assumindo que a aba tem o mesmo nome

# # --- THRESHOLDS ---
# DISTANCE_THRESHOLD_KM = 60.0 # Você ajustou para 60.0
# EMISSION_PERCENT_THRESHOLD = 0.15

# # --- Função limpar_id ---
# def limpar_id(id_series):
#     return id_series.astype(str).str.replace(r'\.0$', '', regex=True).str.strip().str.zfill(21)

# # --- Função para Carregar e Preparar um Arquivo Baseline ---
# def carregar_baseline_excel(filepath, sheetname, header_index=0):
#     print(f"   -> Carregando Baseline: '{os.path.basename(filepath)}' (Aba: '{sheetname}')...")
#     try:
#         df_base = pd.read_excel(filepath, sheet_name=sheetname, header=header_index)
#         df_base.columns = df_base.columns.str.strip() # Limpa nomes das colunas
        
#         # Verifica colunas essenciais
#         colunas_necessarias = ['Identificador do processo de viagem', 'Distância (km)', 'Emissões']
#         if not all(col in df_base.columns for col in colunas_necessarias):
#              # Tenta renomear se encontrar 'identificador'
#              cols_similares_id = [col for col in df_base.columns if 'identificador' in col.lower()]
#              if cols_similares_id:
#                   df_base.rename(columns={cols_similares_id[0]: 'Identificador do processo de viagem'}, inplace=True)
#                   print(f"      - Coluna ID similar '{cols_similares_id[0]}' renomeada.")
             
#              # Verifica novamente
#              col_faltante = next((col for col in colunas_necessarias if col not in df_base.columns), "Desconhecida")
#              if col_faltante != "Desconhecida":
#                   print(f"      ⚠️ Aviso: Coluna '{col_faltante}' não encontrada em '{os.path.basename(filepath)}'.")
#                   # Adiciona coluna faltante com NaN se for 'Distância (km)' ou 'Emissões' para evitar erro no groupby
#                   if col_faltante in ['Distância (km)', 'Emissões']: df_base[col_faltante] = np.nan
#                   else: raise KeyError(f"Coluna ID '{col_faltante}' ainda não encontrada.")
                     
#         # Padroniza ID e limpa dados
#         df_base['Identificador do processo de viagem'] = limpar_id(df_base['Identificador do processo de viagem'])
        
#         # Agrega por ID (para o caso de um ID aparecer múltiplas vezes na mesma planilha)
#         df_base_agg = df_base.groupby('Identificador do processo de viagem').agg({
#             'Distância (km)': 'sum',
#             'Emissões': 'sum'
#         }).reset_index()
        
#         print(f"      - Carregado e agregado. Encontradas {len(df_base_agg)} viagens únicas.")
#         return df_base_agg
        
#     except FileNotFoundError:
#         print(f"   ❌ ERRO DE ARQUIVO: Baseline não encontrado em '{filepath}'.")
#         return pd.DataFrame(columns=['Identificador do processo de viagem', 'Distância (km)', 'Emissões']) # Retorna DF vazio com colunas
#     except Exception as e_base:
#         print(f"   ❌ Erro ao carregar baseline '{os.path.basename(filepath)}': {e_base}")
#         return pd.DataFrame(columns=['Identificador do processo de viagem', 'Distância (km)', 'Emissões']) # Retorna DF vazio

# try:
#     print("🔄 Carregando arquivos para comparação...")

#     # --- Carregar df_ufpb (Notebook) ---
#     try:
#         df_ufpb = pd.read_csv(df_ufpb_path)
#         print(f"✅ Arquivo Notebook carregado ({len(df_ufpb)} viagens).")
#         if 'Identificador do processo de viagem' not in df_ufpb.columns:
#              cols_similares_ufpb = [col for col in df_ufpb.columns if 'identificador' in col.lower()]
#              if cols_similares_ufpb: df_ufpb.rename(columns={cols_similares_ufpb[0]: 'Identificador do processo de viagem'}, inplace=True)
#              else: raise KeyError("'Identificador do processo de viagem' não encontrado em df_ufpb.")
#         df_ufpb['Identificador do processo de viagem'] = limpar_id(df_ufpb['Identificador do processo de viagem'])
#     except Exception as e_ufpb:
#         print(f"❌ Erro ao carregar ou verificar df_ufpb: {e_ufpb}")
#         raise

#     # --- Carregar df_itinerarios (Opcional) ---
#     try:
#         df_itinerarios = pd.read_csv(arquivo_itinerarios)
#         print(f"✅ Arquivo Itinerários carregado.")
#         if 'Identificador do processo de viagem' not in df_itinerarios.columns:
#              cols_similares_itin = [col for col in df_itinerarios.columns if 'identificador' in col.lower()]
#              if cols_similares_itin: df_itinerarios.rename(columns={cols_similares_itin[0]: 'Identificador do processo de viagem'}, inplace=True)
#         df_itinerarios['Identificador do processo de viagem'] = limpar_id(df_itinerarios['Identificador do processo de viagem'])
#     except FileNotFoundError: print(f"⚠️ Aviso: Arquivo de Itinerários não encontrado."); df_itinerarios = pd.DataFrame()
#     except Exception as e_itin: print(f"⚠️ Erro ao carregar df_itinerarios: {e_itin}."); df_itinerarios = pd.DataFrame()


#     # --- Carregar AMBOS os Baselines e Concatenar ---
#     print("\n🔄 Carregando Baselines (2023.1 e 2023.2)...")
#     df_baseline_2023_1 = carregar_baseline_excel(local_baseline_2023_1, sheet_name_baseline_2023_1, header_index=0)
#     df_baseline_2023_2 = carregar_baseline_excel(local_baseline_2023_2, sheet_name_baseline_2023_2, header_index=0)
    
#     # Junta os dois baselines
#     df_baseline_completo = pd.concat([df_baseline_2023_1, df_baseline_2023_2], ignore_index=True)
    
#     # Agrega novamente caso o MESMO ID apareça em AMBOS os arquivos
#     print("\n🔄 Agregando Baseline completo (2023.1 + 2023.2)...")
#     df_baseline_agg_total = df_baseline_completo.groupby('Identificador do processo de viagem').agg({
#         'Distância (km)': 'sum',
#         'Emissões': 'sum'
#     }).reset_index()
#     df_baseline_agg_total = df_baseline_agg_total.rename(columns={'Distância (km)': 'Distancia_Baseline', 'Emissões': 'Emissoes_Baseline'})
#     print(f"✅ Baseline total agregado. Encontradas {len(df_baseline_agg_total)} viagens únicas nos dois arquivos.")

    
#     # --- Diagnóstico e Preparação ---
#     ids_notebook = set(df_ufpb['Identificador do processo de viagem'].unique())
#     ids_baseline_total = set(df_baseline_agg_total['Identificador do processo de viagem'].unique())
    
#     ids_em_comum = ids_notebook.intersection(ids_baseline_total)
#     ids_so_no_notebook = ids_notebook - ids_baseline_total
#     ids_so_no_baseline = ids_baseline_total - ids_notebook

#     print(f"\n🔬 Diagnóstico (Notebook vs Baseline Total):")
#     print(f"   - {len(ids_em_comum)} IDs em comum (serão comparados).")
#     print(f"   - {len(ids_so_no_notebook)} IDs encontrados APENAS no Notebook (serão salvos em 'nao_encontrados_no_baseline.csv').")
#     print(f"   - {len(ids_so_no_baseline)} IDs encontrados APENAS no Baseline (serão ignorados na comparação).")

#     # Renomeia colunas do df_ufpb
#     df_ufpb = df_ufpb.rename(columns={'Distância (GCD)': 'Distancia_Notebook', 'Emissões (KgCO2eq)': 'Emissoes_Notebook'})
#     df_ufpb_subset = df_ufpb[['Identificador do processo de viagem', 'Distancia_Notebook', 'Emissoes_Notebook']]

#     df_itinerarios_subset = pd.DataFrame()
#     if not df_itinerarios.empty and all(c in df_itinerarios for c in ['Identificador do processo de viagem', 'Distancia_Total_Viagem']):
#          df_itinerarios_subset = df_itinerarios[['Identificador do processo de viagem', 'Distancia_Total_Viagem']].rename(columns={'Distancia_Total_Viagem':'Distancia_Itinerario'})

#     # --- Junção (Merge) ---
#     print("\n🔄 Realizando a junção (merge) dos dados em comum...")
#     # Usa how='inner' para comparar apenas os IDs em comum
#     df_merged = pd.merge(df_ufpb_subset, df_baseline_agg_total, on='Identificador do processo de viagem', how='inner') 

#     if not df_itinerarios_subset.empty:
#         df_merged = pd.merge(df_merged, df_itinerarios_subset, on='Identificador do processo de viagem', how='left')
#     else:
#         df_merged['Distancia_Itinerario'] = np.nan

#     # --- Geração do Resultado (Comparação) ---
#     if df_merged.empty:
#         print("\n❌ AVISO: Nenhuma correspondência encontrada na comparação.")
#     else:
#         # ... (Cálculo das diferenças - sem alterações) ...
#         df_merged['Diferenca_Distancia'] = df_merged['Distancia_Notebook'] - df_merged['Distancia_Baseline']
#         df_merged['Diferenca_Emissoes'] = df_merged['Emissoes_Notebook'] - df_merged['Emissoes_Baseline']
#         df_merged['Abs_Dif_Distancia'] = df_merged['Diferenca_Distancia'].abs()
#         df_merged['Perc_Dif_Emissoes'] = np.where(df_merged['Emissoes_Baseline'].notna() & (df_merged['Emissoes_Baseline'] != 0), (df_merged['Diferenca_Emissoes'] / df_merged['Emissoes_Baseline']).abs(), np.where(df_merged['Emissoes_Notebook'].notna() & (df_merged['Emissoes_Notebook'] != 0), np.inf, 0))
#         if 'Distancia_Itinerario' in df_merged.columns: df_merged['Dif_Dist_Notebook_vs_Itinerario'] = df_merged['Distancia_Notebook'] - df_merged['Distancia_Itinerario']
#         else: df_merged['Dif_Dist_Notebook_vs_Itinerario'] = np.nan

#         caminho_saida_completo = 'diferenca_FINAL_completa.csv'
#         df_merged.round(4).to_csv(caminho_saida_completo, index=False)
#         print(f"\n✅ Comparação completa (dados em comum) salva em: '{caminho_saida_completo}' ({len(df_merged)} viagens)")

#         # --- FILTRAGEM DAS DISCREPÂNCIAS ---
#         print(f"\n🔄 Filtrando viagens com |Diferença_Distância| > {DISTANCE_THRESHOLD_KM} km OU |Perc_Dif_Emissoes| > {EMISSION_PERCENT_THRESHOLD*100:.0f}%...")
#         filtro_distancia = df_merged['Abs_Dif_Distancia'] > DISTANCE_THRESHOLD_KM
#         filtro_emissao = (df_merged['Perc_Dif_Emissoes'] > EMISSION_PERCENT_THRESHOLD) & (df_merged['Perc_Dif_Emissoes'] != np.inf) & (df_merged['Perc_Dif_Emissoes'].notna())
#         df_discrepancias = df_merged[filtro_distancia | filtro_emissao].copy()

#         if df_discrepancias.empty:
#             print(f"✅ Nenhuma viagem encontrada com discrepâncias significativas.")
#         else:
#             print(f"\n--- {len(df_discrepancias)} VIAGENS COM DISCREPÂNCIAS SIGNIFICATIVAS ---")
#             # ... (Código para imprimir e salvar discrepâncias - sem alterações) ...
#             df_discrepancias_sorted = df_discrepancias.sort_values(by='Abs_Dif_Distancia', ascending=False)
#             colunas_exibir = ['Identificador do processo de viagem', 'Distancia_Notebook', 'Distancia_Baseline', 'Diferenca_Distancia', 'Emissoes_Notebook', 'Emissoes_Baseline', 'Diferenca_Emissoes', 'Perc_Dif_Emissoes']
#             df_discrepancias_sorted['Perc_Dif_Emissoes_Str'] = df_discrepancias_sorted['Perc_Dif_Emissoes'].apply(lambda x: f"{x*100:.1f}%" if pd.notna(x) and np.isfinite(x) else ('Infinita' if x == np.inf else 'N/A'))
#             print(df_discrepancias_sorted[colunas_exibir[:-1] + ['Perc_Dif_Emissoes_Str']].round(2).to_string())
#             caminho_saida_discrepancias = 'diferenca_SIGNIFICATIVAS.csv'
#             df_discrepancias_sorted[colunas_exibir].round(4).to_csv(caminho_saida_discrepancias, index=False)
#             print(f"\n✅ Viagens com discrepâncias significativas salvas em: '{caminho_saida_discrepancias}'")
            
#             if 'Dif_Dist_Notebook_vs_Itinerario' in df_merged.columns and df_merged['Dif_Dist_Notebook_vs_Itinerario'].notna().any():
#                  max_dif_interna = df_merged['Dif_Dist_Notebook_vs_Itinerario'].abs().max()
#                  if max_dif_interna < 0.01: print(f"\n✅ Verificação Interna OK (Diferença máx: {max_dif_interna:.4f} km).")
#                  else: print(f"\n⚠️ Verificação Interna FALHOU (Diferença máx: {max_dif_interna:.4f} km).")

#     # --- NOVO: SALVAR VIAGENS QUE SÓ ESTÃO NO NOTEBOOK ---
#     if ids_so_no_notebook:
#         print(f"\n🔄 Salvando {len(ids_so_no_notebook)} viagens que estão no notebook mas não no baseline...")
#         df_nao_encontradas = df_ufpb[df_ufpb['Identificador do processo de viagem'].isin(ids_so_no_notebook)].copy()
#         caminho_saida_nao_encontradas = 'viagens_nao_encontradas_no_baseline.csv'
#         df_nao_encontradas.to_csv(caminho_saida_nao_encontradas, index=False)
#         print(f"✅ Arquivo com viagens não encontradas no baseline salvo em: '{caminho_saida_nao_encontradas}'")
#     else:
#         print("\n✅ Todas as viagens do notebook foram encontradas no baseline.")

# except FileNotFoundError as e:
#      print(f"❌ ERRO DE ARQUIVO: {e}")
# except KeyError as e:
#      print(f"❌ ERRO DE COLUNA: {e}. Verifique se o 'header' está correto e se as colunas necessárias existem.")
# except Exception as e:
#     print(f"❌ Ocorreu um erro inesperado: {e}")

In [159]:
# copia = df_ufpb[df_ufpb["Identificador do processo de viagem"].isin(df_nao_encontradas["Identificador do processo de viagem"])]
# copia

In [160]:
# copia = trecho_df[trecho_df["Identificador do processo de viagem"].isin(df_nao_encontradas["Identificador do processo de viagem"])]
# copia

salvei com maior de 100 km de diferença

In [161]:
# df_discrepancias_sorted

## verificando ids 

In [162]:
# # --- CÉLULA DE INVESTIGAÇÃO DE TRECHOS ESPECÍFICOS ---
# import pandas as pd

# # IDs das viagens com as NOVAS grandes discrepâncias
# # id_positivo_novo = '0000000000018697949' 
# # id_negativo_novo = '000000000000018842529' 
# id_positivo_novo = '000000000000018697949' 
# id_negativo_novo = '000000000000018720751' 
# ids_para_verificar = [id_positivo_novo, id_negativo_novo]

# print("\n--- Investigando Trechos para IDs com Grandes Discrepâncias ---")

# try:
#     if 'trecho_df' in locals() and not trecho_df.empty:
#         trecho_df['Identificador do processo de viagem'] = trecho_df['Identificador do processo de viagem'].astype(str).str.zfill(21)
        
#         for trip_id in ids_para_verificar:
#             print(f"\n--- Detalhes para Viagem ID: {trip_id} ---")
#             trechos_viagem = trecho_df[trecho_df['Identificador do processo de viagem'] == trip_id]
            
#             if not trechos_viagem.empty:
#                 colunas_mostrar = ['Origem - Cidade', 'Destino - Cidade', 'Distância (GCD)']
#                 if 'Latitude_Origem' in trechos_viagem.columns:
#                      colunas_mostrar.extend(['Latitude_Origem', 'Longitude_Origem', 'Latitude_Destino', 'Longitude_Destino'])
                     
#                 print("Trechos encontrados e suas distâncias (GCD calculadas pelo notebook):")
#                 with pd.option_context('display.max_rows', None): 
#                     print(trechos_viagem[colunas_mostrar].round(2).to_string())
                
#                 soma_distancias = trechos_viagem['Distância (GCD)'].sum()
#                 print(f"\nSoma das Distâncias (Notebook): {soma_distancias:.2f} km")
                
#                 # Busca o valor do Baseline (carregando o arquivo de diferenças)
#                 try:
#                      df_diff = pd.read_csv('diferenca_FINAL_completa.csv') 
#                      df_diff['Identificador do processo de viagem'] = limpar_id(df_diff['Identificador do processo de viagem']) 
#                      baseline_info = df_diff[df_diff['Identificador do processo de viagem'] == trip_id]
#                      if not baseline_info.empty:
#                           dist_baseline = baseline_info['Distancia_Baseline'].iloc[0]
#                           print(f"Distância no Baseline (Excel): {dist_baseline:.2f} km")
#                           print(f"Diferença Registrada: {soma_distancias - dist_baseline:.2f} km")
#                      else:
#                           print("Distância do Baseline não encontrada no arquivo de diferenças.")
#                 except FileNotFoundError:
#                      print("Arquivo 'diferenca_FINAL_completa.csv' não encontrado.")
#                 except NameError:
#                      print("Função 'limpar_id' não definida. Execute a Etapa 6/Célula 14 primeiro.")
#                 except Exception as e_diff:
#                      print(f"Erro ao buscar dados do baseline: {e_diff}")
#             else:
#                 print(f"Nenhum trecho encontrado para este ID no 'trecho_df' atual.")
#             print("-" * 50)
            
#     else:
#         print("❌ ERRO: DataFrame 'trecho_df' não encontrado ou vazio. Execute as células anteriores.")
# except Exception as e:
#     print(f"❌ Ocorreu um erro inesperado durante a investigação: {e}")

# comparação Global

In [163]:
# # --- ETAPA 7: COMPARAÇÃO DOS TOTAIS (NOTEBOOK vs BASELINE) ---
# import pandas as pd
# import numpy as np

# print("\n--- Comparando Totais (Soma das Distâncias e Emissões) ---")

# # Nome do arquivo com os dados mesclados (resultado da Etapa 6)
# arquivo_comparacao_completa = 'diferenca_FINAL_completa.csv' 

# try:
#     # Carrega o DataFrame com os dados mesclados
#     df_merged = pd.read_csv(arquivo_comparacao_completa)
#     print(f"✅ Arquivo '{arquivo_comparacao_completa}' carregado com {len(df_merged)} viagens.")

#     if not df_merged.empty:
#         # 1. Calcular Totais
#         total_dist_notebook = df_merged['Distancia_Notebook'].sum()
#         total_dist_baseline = df_merged['Distancia_Baseline'].sum()
#         total_emiss_notebook = df_merged['Emissoes_Notebook'].sum()
#         total_emiss_baseline = df_merged['Emissoes_Baseline'].sum()

#         # 2. Calcular Diferenças Totais
#         diff_dist_total = total_dist_notebook - total_dist_baseline
#         diff_emiss_total = total_emiss_notebook - total_emiss_baseline

#         # 3. Calcular Diferenças Percentuais Totais (trata divisão por zero)
#         perc_diff_dist = (diff_dist_total / total_dist_baseline * 100) if total_dist_baseline != 0 else np.nan
#         perc_diff_emiss = (diff_emiss_total / total_emiss_baseline * 100) if total_emiss_baseline != 0 else np.nan

#         # 4. Apresentar Resultados
#         print(f"\n--- Comparativo dos Totais (para as {len(df_merged)} viagens em comum) ---")
#         print(f"                                   |   Notebook   |    Baseline  |  Diferença   | Diferença (%)")
#         print("-" * 75)
#         print(f"Distância Total (km)                | {total_dist_notebook:12,.2f} | {total_dist_baseline:12,.2f} | {diff_dist_total:12,.2f} |    {perc_diff_dist:7.2f}%")
#         print(f"Emissões Totais (KgCO2eq)           | {total_emiss_notebook:12,.2f} | {total_emiss_baseline:12,.2f} | {diff_emiss_total:12,.2f} |    {perc_diff_emiss:7.2f}%")
#         print("-" * 75)

#         # Análise breve (ajuste conforme os valores percentuais que você obter)
#         if abs(perc_diff_dist) < 5:
#              print(f"\nAnálise Distância: Os totais de distância estão bem alinhados (diferença < {perc_diff_dist}).")
#         else:
#              print("\nAnálise Distância: Há uma diferença notável nos totais de distância.")
             
#         if abs(perc_diff_emiss) < 10: # Ajuste este threshold se necessário
#              print(f"Análise Emissões: Os totais de emissões estão emissoes < {perc_diff_emiss}.")
#         elif perc_diff_emiss > 100: # Se for mais que o dobro
#               print("Análise Emissões: As emissões totais do Notebook são significativamente maiores (mais que o dobro), provavelmente devido à inclusão de Uplift e RFI.")
#         elif perc_diff_emiss < -50: # Se for menos da metade
#                print("Análise Emissões: As emissões totais do Notebook são significativamente menores, indicando possível omissão de fatores ou EF diferente.")
#         else:
#              print("Análise Emissões: Há uma diferença considerável nos totais de emissões a ser investigada (provavelmente devido a Fatores de Emissão diferentes).")

#     else:
#         print("⚠️ O DataFrame mesclado está vazio. Não é possível calcular os totais.")

# except FileNotFoundError:
#     print(f"❌ ERRO: Arquivo '{arquivo_comparacao_completa}' não encontrado. Execute a Etapa 6 primeiro.")
# except KeyError as e:
#     print(f"❌ ERRO DE COLUNA: Coluna {e} não encontrada no arquivo '{arquivo_comparacao_completa}'. Verifique a Etapa 6.")
# except Exception as e:
#     print(f"❌ Ocorreu um erro inesperado: {e}")